## Test July 06 Cluster Shape Alignment

In [97]:
from pathlib import Path 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_07") 
DATA_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_07")
LABELS_PATH = DATA_DIR / "11_consensus_labels.csv" 
PANEL_NORM_PATH = DATA_DIR / "02_preprocess_normalized.csv"

labels = pd.read_csv(LABELS_PATH) 
panel_norm = pd.read_csv(PANEL_NORM_PATH, index_col=0, parse_dates=True ) 
panel_norm.index.name = "Time" 
labels["term"] = labels["term"].astype(str) 
labels["cluster"] = labels["cluster"].astype(int) # Keep only terms that exist in normalized panel 
labels = labels[labels["term"].isin(panel_norm.columns)].copy() 
print(f"Terms in labels after matching panel columns: {len(labels)}")

# ----------------------------------------------------------------------------
# Plot time series of top 20 most stable words for each cluster
# ----------------------------------------------------------------------------

TOP_N = 20

# Rank words within each cluster by stability_score
top_terms = (
    labels
    .sort_values(
        ["cluster", "stability_score"],
        ascending=[True, False]
    )
    .groupby("cluster", group_keys=False)
    .head(TOP_N)
    .copy()
)

# Keep only terms that exist in normalized panel
top_terms = top_terms[top_terms["term"].isin(panel_norm.columns)].copy()

# Plot each cluster separately
for cluster_id, group in top_terms.groupby("cluster"):

    cluster_terms = group["term"].tolist()
    cluster_ts = panel_norm[cluster_terms]

    fig_path = OUTPUT_DIR / f"cluster_{cluster_id}_top20_stable_timeseries.png"

    plt.figure(figsize=(15, 7))

    for term in cluster_terms:
        plt.plot(
            cluster_ts.index,
            cluster_ts[term],
            linewidth=1.2,
            alpha=0.8,
            label=term
        )

    plt.title(
        f"Cluster {cluster_id}: Top {len(cluster_terms)} Most Stable Words",
        fontsize=15
    )
    plt.xlabel("Time")
    plt.ylabel("Normalized Search Interest")
    plt.grid(alpha=0.3)

    plt.legend(
        loc="upper left",
        bbox_to_anchor=(1.01, 1),
        fontsize=8,
        frameon=False
    )

    plt.tight_layout()

    plt.savefig(
        fig_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(f"Saved cluster {cluster_id} plot to: {fig_path}")

Terms in labels after matching panel columns: 830
Saved cluster 0 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_0_top20_stable_timeseries.png
Saved cluster 1 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_1_top20_stable_timeseries.png
Saved cluster 2 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_2_top20_stable_timeseries.png
Saved cluster 3 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_3_top20_stable_timeseries.png
Saved cluster 4 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_4_top20_stable_timeseries.png
Saved cluster 5 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_5_top20_stable_timeseries.png
Saved cluster 6 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_6_top20_stable_timeseries.png
Saved cluster 7 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_7_top20_stable_timeseries.png
Saved cluster 8 plot to: C:\Python\CSUREMM\output\sax_tests_july_07\cluster_8_top20_stable_timeseries.

## Event Attention-Curve Clustering with Sliding SAX

```text
Google Trends chunk files
        ↓
overlap-rolling stitched daily series
        ↓
quality diagnostics
        ↓
light denoising
        ↓
detrending
        ↓
robust per-series normalization
        ↓
sliding SAX parameter tuning
        ├── window_size
        ├── step_size
        ├── PAA word length w / n_segments
        └── alphabet_size a
        ↓
SAX feature matrix
        ↓
KMeans / Ward clustering
        ↓
cluster interpretation tables and plots
```

The notebook **does not re-query Google Trends**. It treats the existing overlap-rolling stitched files under
`DATA_DIR/<query>/stitched/` as the canonical daily input.

### External references used for this revision

1. `candicedjorno/replication-restoring-google-trends` motivates the preprocessing order: cluster or group related terms, smooth/denoise noisy Google Trends curves, then detrend before modeling. In this notebook, that logic is adapted to an unsupervised SAX clustering setting rather than forecasting.
   - Repository: https://github.com/candicedjorno/replication-restoring-google-trends
   - Relevant ideas/files: `scripts/01_clustering/hierarchical_clustering.ipynb`, `scripts/02_denoising/gt_denoising.R`, `scripts/03_detrending/detrending.py`

2. `trendecon/trendecon` motivates careful construction of consistent long daily Google Trends series. Since the current project has already downloaded chunk-level daily data and stitched it with overlap scaling, this notebook does **not** implement trendEcon's multi-frequency reconstruction. It instead adds diagnostics and post-stitch preprocessing.
   - Repository: https://github.com/trendecon/trendecon
   - Documentation: https://trendecon.github.io/trendecon/reference/ts_gtrends_mwd.html
   - Relevant idea: robust long daily Google Trends series from daily/weekly/monthly harmonization; here used as conceptual guidance, not as executable dependency.

### Main revision choices

- Removed the consensus-matrix / graph-clustering tuning from the core workflow.
- Kept only the three SAX tuning families requested:
  - `window_size`
  - `step_size`
  - `PAA word length w`, implemented as `n_segments`
  - `alphabet_size a`
- Separated **representation tuning** from **cluster-count selection**.
- Added robust preprocessing functions: small-gap interpolation, light denoising, detrending, and robust z-scoring.
- Added output notes so every selected parameter setting is saved for reporting.


### Efficiency revision added

This version also improves computation efficiency in the SAX tuning stage by:

- caching clean NumPy arrays once instead of repeatedly converting each pandas Series;
- caching sliding windows by `(window_size, step)` so the same windows are reused across PAA and alphabet settings;
- computing PAA segment means with cumulative sums instead of Python loops over every window;
- vectorizing SAX discretization with `np.searchsorted`;
- using sampled silhouette scores during grid search to avoid expensive pairwise-distance calculations on large term sets;
- using a smaller `n_init` for tuning and reserving heavier `KMeans` only for the final selected model.


In [121]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# CONFIG -- edit for your local machine
# -----------------------------------------------------------------------------

DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_07")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Current folder structure:
# DATA_DIR/
#     query_metadata.csv
#     weather/
#         chunks/
#         diagnostics/
#         stitched/
#             gt_stitched_weather_2022-01-01_2026-05-31.csv
STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed_*.csv"

# Preprocessing controls.
MAX_INTERPOLATE_GAP = 7       # fill missing gaps up to one week
DENOISE_WINDOW = 15           # odd integer; light smoothing only
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91           # 13-week rolling median trend

# SAX tuning grid. PAA word length w is named n_segments in the code.
WINDOW_GRID = [60, 90, 180] # removed 365 days because it was too coarse for the 2022-2026 data
STEP_FRACTIONS = [0.25]       # step = round(window_size * fraction)
PAA_WORD_LENGTH_GRID = [8, 12, 16, 24, 32, 52]
ALPHABET_GRID = [3, 4, 5, 6]

# Clustering grid. This is evaluated after a SAX representation is built.
K_GRID = range(2, 8)
RANDOM_STATE = 42

# --- Anti-degeneracy / balance controls -------------------------------------
# A k=2 split that isolates a handful of outlier terms from everything else
# is statistically "clean" (high silhouette) but not interpretable. These
# controls stop the autoselection stage from rewarding that kind of split.
MIN_CLUSTER_FRACTION = 0.08   # smallest allowed cluster, as a fraction of n_terms
BALANCE_METRIC = "norm_entropy"  # "norm_entropy" or "min_frac"

# --- Multi-resolution (layered) clustering -----------------------------------
LAYER_KS = (3, 6, 10)   # coarse -> meso -> fine cuts of the same dendrogram

# --- Consensus clustering -----------------------------------------------------
CONSENSUS_K_RANGE = range(2, 11)
CONSENSUS_N_RESAMPLES = 200
CONSENSUS_SUBSAMPLE_FRAC = 0.8

# Optional spike features summarize the full preprocessed curve in addition to symbolic shape.
INCLUDE_SPIKE_FEATURES = False

# Efficiency controls for tuning.
# Silhouette is O(n_terms^2); sampling makes grid search much faster on large panels.
TUNING_KMEANS_N_INIT = 10
FINAL_KMEANS_N_INIT = 50
SILHOUETTE_SAMPLE_SIZE = 500
USE_WINDOW_CACHE = True


# Functions

## 1. Load stitched daily series

The file of interest is the stitched daily series produced from the `chunks/` folder. The raw chunk files remain important for diagnostics, but the clustering input is the stitched output.

This is the practical compromise between the two external references:

- trendEcon emphasizes that long Google Trends series need careful scaling across time windows.
- The current project already performed overlap-rolling stitching, so we avoid re-querying or re-normalizing every chunk.
- The Djorno replication motivates the next stage: statistical preprocessing before modeling/clustering.


In [80]:
def load_all_series(
    data_dir: Path,
    stitched_subdir: str = STITCHED_SUBDIR,
    filename_glob: str = STITCHED_GLOB,
) -> dict[str, pd.Series]:
    """
    Load one stitched daily Google Trends CSV per query folder.

    Expected format:
        DATA_DIR/<query>/stitched/gt_stitched_<query>_<start>_<end>.csv

    The first non-Time column is treated as the query's value column.
    """
    series_dict: dict[str, pd.Series] = {}
    failed: list[tuple[str, str]] = []

    for folder in sorted(data_dir.iterdir()):
        if not folder.is_dir():
            continue

        stitched_dir = folder / stitched_subdir
        candidates = sorted(stitched_dir.glob(filename_glob))

        if not candidates:
            failed.append((folder.name, f"missing stitched csv in {stitched_dir}"))
            continue

        fpath = candidates[-1]

        try:
            df = pd.read_csv(fpath, parse_dates=["Time"])
            value_cols = [c for c in df.columns if c != "Time"]
            if not value_cols:
                raise ValueError("no value column found")

            value_col = value_cols[0]
            ts = (
                df[["Time", value_col]]
                .dropna(subset=["Time"])
                .drop_duplicates("Time")
                .sort_values("Time")
                .set_index("Time")[value_col]
                .astype(float)
            )
            ts.name = folder.name
            series_dict[folder.name] = ts

        except Exception as e:
            failed.append((folder.name, str(e)))

    print(f"Loaded {len(series_dict)} stitched series.")
    if failed:
        print(f"Failed to load {len(failed)} folders:")
        for name, err in failed[:25]:
            print(f"  - {name}: {err}")
        if len(failed) > 25:
            print(f"  ... {len(failed) - 25} more")

    return series_dict


def build_panel(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """Align all query series on a common daily date index."""
    if not series_dict:
        raise ValueError("series_dict is empty")

    panel = pd.DataFrame(series_dict).sort_index()
    full_index = pd.date_range(panel.index.min(), panel.index.max(), freq="D")
    return panel.reindex(full_index)


def basic_quality_report(panel: pd.DataFrame) -> pd.DataFrame:
    """Basic post-stitch diagnostics per query."""
    rows = []
    for col in panel.columns:
        x = panel[col]
        rows.append({
            "query": col,
            "n_days_total": len(x),
            "n_nonmissing": int(x.notna().sum()),
            "missing_share": float(x.isna().mean()),
            "zero_share": float((x == 0).mean(skipna=True)),
            "min": float(x.min(skipna=True)),
            "max": float(x.max(skipna=True)),
            "mean": float(x.mean(skipna=True)),
            "std": float(x.std(skipna=True)),
            "range": float(x.max(skipna=True) - x.min(skipna=True)),
        })
    return pd.DataFrame(rows).sort_values("query")


## 2. Post-stitch preprocessing

The preprocessing layer adapts the Djorno et al. logic to this project:

1. fill short missing gaps;
2. apply light denoising, so SAX does not overreact to one-day noise;
3. remove slow-moving trend, so clustering focuses on attention shape;
4. robustly normalize each query once over the full daily series.

This normalization is **not per chunk** and **not per SAX window**. Window-level z-normalization still occurs inside SAX so each local word describes shape rather than absolute level.


In [119]:
def interpolate_small_gaps(s: pd.Series, max_gap: int = MAX_INTERPOLATE_GAP) -> pd.Series:
    """Interpolate short missing gaps without filling long absent regions."""
    return s.astype(float).interpolate(
        method="time",
        limit=max_gap,
        limit_direction="both",
    )


def denoise_series(
    s: pd.Series,
    window: int = DENOISE_WINDOW,
    polyorder: int = DENOISE_POLYORDER,
) -> pd.Series:
    """Light Savitzky-Golay denoising. Falls back to rolling median for short series."""
    x = s.astype(float)
    y = x.copy()
    valid = x.dropna()
    if len(valid) < 5:
        return x
    win = min(window, len(valid) if len(valid) % 2 == 1 else len(valid) - 1)
    if win <= polyorder + 2:
        return x.rolling(7, center=True, min_periods=1).median()
    if win % 2 == 0:
        win -= 1
    filled = interpolate_small_gaps(x).ffill().bfill()
    smoothed = savgol_filter(filled.values, window_length=win, polyorder=polyorder)
    y.loc[:] = smoothed
    y[x.isna()] = np.nan
    return y.rename(s.name)


def detrend_series(s: pd.Series, window: int = DETREND_WINDOW) -> pd.Series:
    """Remove slow trend using a centered rolling median."""
    x = s.astype(float)
    trend = x.rolling(window=window, center=True, min_periods=max(7, window // 4)).median()
    trend = trend.bfill().ffill()
    return (x - trend).rename(s.name)


def robust_zscore_series(
    s: pd.Series,
    mad_floor_frac: float = 0.05,
    zero_share_threshold: float = 0.30,
    mad_ratio_threshold: float = 0.02,
    clip_mad_multiple: float = 4.0,
) -> pd.Series:
    """
    Robust per-query normalization using median and MAD.

    Only applies a MAD floor + winsorization for series that actually look
    like the degenerate "near-empty baseline + rare spike" case (high
    zero-share and/or a MAD that is tiny relative to the series' own spread).
    Well-behaved series pass through with plain MAD z-scoring, untouched.
    """
    x = s.astype(float)
    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)
        if pd.isna(std) or std == 0:
            return pd.Series(np.zeros(len(x)), index=x.index, name=x.name)
        return ((x - x.mean(skipna=True)) / std).rename(x.name)

    # tail-aware spread, used only to detect/floor the degenerate case
    p05, p95 = x.quantile(0.05), x.quantile(0.95)
    wide_spread = (p95 - p05) / 3.29

    zero_share = float((x == 0).mean())
    mad_ratio = (mad / wide_spread) if wide_spread > 0 else 1.0

    is_degenerate = mad_ratio < 0.05

    if not is_degenerate:
        # normal path: untouched MAD z-score, no floor, no clip
        return (0.6745 * (x - med) / mad).rename(x.name)

    # degenerate path: floor the scale, then winsorize relative to the
    # (floored) scale itself rather than an absolute cutoff, so the clip
    # adapts to each series instead of chopping every series at the same point
    mad_floored = max(mad, mad_floor_frac * wide_spread) if wide_spread > 0 else mad
    z = (0.6745 * (x - med) / mad_floored).rename(x.name)
    return z.clip(lower=-clip_mad_multiple, upper=clip_mad_multiple)

def soft_cap_series(s: pd.Series, cap: float = 10.0) -> pd.Series:
    return (cap * np.tanh(s / cap)).rename(s.name)

def soft_cap_panel(df: pd.DataFrame, cap: float = 10.0) -> pd.DataFrame:
    return df.apply(soft_cap_series, axis=0, cap=cap)

def preprocess_panel(panel: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """
    Apply the current preprocessing pipeline to all columns.

    Returns
    -------
    panel_preprocessed:
        daily panel after interpolation, denoising, detrending, and robust z-score.
    stages:
        intermediate panels useful for diagnostics and plotting.
    """
    filled = panel.apply(interpolate_small_gaps, axis=0)
    denoised = filled.apply(denoise_series, axis=0)
    detrended = denoised.apply(detrend_series, axis=0)
    normalized = detrended.apply(robust_zscore_series, axis=0)
    normalized_soft = soft_cap_panel(
         normalized,
         cap=10.0
    )
    
    stages = {
        "filled": filled,
        "denoised": denoised,
        "detrended": detrended,
        "normalized": normalized,
        "soft": normalized_soft
    }
    return normalized, stages

## 3. Sliding SAX representation

The three retained tuning dimensions are implemented here:

- `window_size`: the local time horizon summarized by one SAX word;
- `step_size`: how often the window moves forward;
- `PAA word length w`: implemented as `n_segments`;
- `alphabet_size a`: number of vertical states.

For example, `window_size = 180`, `step = 30`, `n_segments = 24`, and `alphabet_size = 4` means each 180-day local episode is compressed into a 24-symbol word using four vertical levels.


In [82]:
def gaussian_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian SAX breakpoints for equiprobable symbolic bins."""
    return stats.norm.ppf(np.linspace(0, 1, alphabet_size + 1)[1:-1])


def valid_sax_setting(window_size: int, step: int, n_segments: int, alphabet_size: int) -> bool:
    """Keep settings interpretable and numerically valid."""
    if window_size < 2 or step < 1 or n_segments < 2 or alphabet_size < 2:
        return False
    if n_segments >= window_size:
        return False
    if window_size / n_segments < 3:
        return False
    return True


def panel_to_array_dict(panel: pd.DataFrame) -> dict[str, np.ndarray]:
    """Convert the normalized panel to compact NumPy arrays once."""
    out: dict[str, np.ndarray] = {}
    for col in panel.columns:
        x = panel[col].dropna().to_numpy(dtype=np.float32)
        if len(x) > 0:
            out[col] = x
    return out


def znormalize_windows_matrix(windows: np.ndarray) -> np.ndarray:
    """Vectorized z-normalization for all sliding windows of one series."""
    mu = windows.mean(axis=1, keepdims=True)
    sigma = windows.std(axis=1, keepdims=True)
    sigma[sigma == 0] = 1.0
    return (windows - mu) / sigma


def paa_matrix(z_windows: np.ndarray, n_segments: int) -> np.ndarray:
    """Vectorized PAA using cumulative sums for all windows at once."""
    n_windows, window_size = z_windows.shape
    bounds = np.linspace(0, window_size, n_segments + 1, dtype=np.int64)
    csum = np.concatenate(
        [np.zeros((n_windows, 1), dtype=z_windows.dtype), np.cumsum(z_windows, axis=1)],
        axis=1,
    )
    seg_sum = csum[:, bounds[1:]] - csum[:, bounds[:-1]]
    seg_len = (bounds[1:] - bounds[:-1]).astype(z_windows.dtype)
    return seg_sum / seg_len


def make_window_cache(
    array_dict: dict[str, np.ndarray],
    settings: list[dict[str, int]],
) -> dict[tuple[int, int], dict[str, np.ndarray]]:
    """Cache sliding windows for every distinct `(window_size, step)` pair."""
    cache: dict[tuple[int, int], dict[str, np.ndarray]] = {}
    pairs = sorted({(s["window_size"], s["step"]) for s in settings})

    for window_size, step in pairs:
        term_windows: dict[str, np.ndarray] = {}
        for term, x in array_dict.items():
            if len(x) < window_size:
                continue
            windows = sliding_window_view(x, window_size)[::step].astype(np.float32, copy=True)
            if len(windows) > 0:
                term_windows[term] = windows
        cache[(window_size, step)] = term_windows

    return cache


def build_sax_feature_matrix_fast(
    array_dict: dict[str, np.ndarray],
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int,
    spike_features: bool = INCLUDE_SPIKE_FEATURES,
    window_cache: dict[tuple[int, int], dict[str, np.ndarray]] | None = None,
) -> pd.DataFrame:
    """
    Fast query-level SAX profile construction.

    Feature design is unchanged:
    - mean symbolic level for each PAA segment across sliding windows;
    - distribution of symbols across the whole query;
    - optional spike features from the normalized preprocessed series.
    """
    if not valid_sax_setting(window_size, step, n_segments, alphabet_size):
        raise ValueError(
            f"Invalid SAX setting: window={window_size}, step={step}, "
            f"n_segments={n_segments}, alphabet={alphabet_size}"
        )

    breakpoints = gaussian_breakpoints(alphabet_size).astype(np.float32)
    cached_windows = None if window_cache is None else window_cache.get((window_size, step))
    feature_rows = {}

    iterator = cached_windows.items() if cached_windows is not None else array_dict.items()

    for term, obj in iterator:
        if cached_windows is not None:
            windows = obj
            x = array_dict[term]
        else:
            x = obj
            if len(x) < window_size:
                continue
            windows = sliding_window_view(x, window_size)[::step].astype(np.float32, copy=True)
            if len(windows) == 0:
                continue

        z = znormalize_windows_matrix(windows)
        seg_means = paa_matrix(z, n_segments)
        symbols = np.searchsorted(breakpoints, seg_means).astype(np.int16)

        row = {f"sax_seg_mean_{i + 1:02d}": float(v) for i, v in enumerate(symbols.mean(axis=0))}

        counts = np.bincount(symbols.ravel(), minlength=alphabet_size).astype(float)
        shares = counts / counts.sum()
        for a, share in enumerate(shares):
            row[f"symbol_share_{a}"] = float(share)

        if spike_features:
            row.update({
                "p95_z": float(np.percentile(x, 95)),
                "p99_z": float(np.percentile(x, 99)),
                "spike_share_z2": float(np.mean(x > 2)),
                "spike_share_z3": float(np.mean(x > 3)),
            })

        feature_rows[term] = row

    return pd.DataFrame.from_dict(feature_rows, orient="index").dropna(axis=0, how="any")


# Backward-compatible alias.
def build_sax_feature_matrix(*args, **kwargs) -> pd.DataFrame:
    return build_sax_feature_matrix_fast(*args, **kwargs)


## 4. Tune the SAX representation

This stage tunes the SAX representation, not the final interpretation. It evaluates each setting using the best available KMeans score over a small `k` grid.

Recommended selection rule:

1. discard settings with too few usable terms;
2. prefer higher silhouette;
3. prefer lower Davies-Bouldin;
4. among close candidates, choose the simpler/more interpretable representation.

This avoids picking a very high-resolution SAX encoding that only wins because it compresses less.


In [83]:
def make_sax_grid(
    window_grid=WINDOW_GRID,
    step_fractions=STEP_FRACTIONS,
    paa_grid=PAA_WORD_LENGTH_GRID,
    alphabet_grid=ALPHABET_GRID,
) -> list[dict[str, int]]:
    """Construct valid SAX settings from the requested parameter families."""
    settings = []
    for window_size in window_grid:
        for frac in step_fractions:
            step = max(1, int(round(window_size * frac)))
            for n_segments in paa_grid:
                for alphabet_size in alphabet_grid:
                    if valid_sax_setting(window_size, step, n_segments, alphabet_size):
                        settings.append({
                            "window_size": int(window_size),
                            "step": int(step),
                            "n_segments": int(n_segments),
                            "alphabet_size": int(alphabet_size),
                        })
    return settings


def cluster_balance_metrics(labels: np.ndarray) -> dict[str, float]:
    """
    Quantify how balanced a clustering is, independent of geometric separation.

    A k=2 solution that isolates 6% of terms as "outliers" and dumps the rest
    into one blob can have excellent silhouette/Davies-Bouldin scores while
    being useless for interpretation. These metrics let the selection rule
    penalize that pattern instead of rewarding it.
    """
    counts = np.bincount(labels)
    counts = counts[counts > 0]
    n = counts.sum()
    k = len(counts)
    min_frac = float(counts.min() / n)
    probs = counts / n
    entropy = float(-(probs * np.log(probs)).sum())
    max_entropy = np.log(k) if k > 1 else 1.0
    norm_entropy = float(entropy / max_entropy) if max_entropy > 0 else 0.0
    return {"min_cluster_frac": min_frac, "norm_entropy": norm_entropy}


def evaluate_clustering_for_features(
    features: pd.DataFrame,
    k_grid=K_GRID,
    random_state: int = RANDOM_STATE,
    n_init: int = TUNING_KMEANS_N_INIT,
    silhouette_sample_size: int | None = SILHOUETTE_SAMPLE_SIZE,
) -> pd.DataFrame:
    """
    Evaluate MiniBatchKMeans across k for one feature matrix.

    Silhouette is expensive because it uses pairwise distances. During tuning,
    this function samples rows when there are many terms.

    In addition to the geometric scores, this now records balance metrics
    (`min_cluster_frac`, `norm_entropy`) for every k so degenerate splits
    (one dominant cluster + a handful of outliers) can be filtered out
    downstream instead of silently winning on silhouette alone.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    rows = []

    sample_size = None
    if silhouette_sample_size is not None and len(features) > silhouette_sample_size:
        sample_size = int(silhouette_sample_size)

    valid_ks = [k for k in k_grid if 2 <= k < len(features)]
    for k in valid_ks:
        labels = MiniBatchKMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=n_init,
            batch_size=min(512, len(features)),
            max_iter=100,
        ).fit_predict(X)

        balance = cluster_balance_metrics(labels)

        rows.append({
            "k": int(k),
            "silhouette": float(silhouette_score(
                X,
                labels,
                sample_size=sample_size,
                random_state=random_state,
            )),
            "davies_bouldin": float(davies_bouldin_score(X, labels)),
            "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
            **balance,
        })

    return pd.DataFrame(rows)


def tune_sax_representation(
    array_dict: dict[str, np.ndarray],
    settings: list[dict[str, int]],
    k_grid=K_GRID,
    spike_features: bool = INCLUDE_SPIKE_FEATURES,
    min_terms: int = 10,
    use_window_cache: bool = USE_WINDOW_CACHE,
    min_cluster_fraction: float = MIN_CLUSTER_FRACTION,
    balance_metric: str = BALANCE_METRIC,
) -> pd.DataFrame:
    """
    Grid-search window, step, PAA word length, and alphabet size efficiently.

    Selection fix: previously the "best" k for a setting was whichever k
    (2..7) maximized silhouette, with no check on whether that k actually
    produced usable clusters. A window=365 / n_segments=8 / alphabet=3
    setting can win at k=2 purely because it isolates a few extreme series
    from one giant undifferentiated blob (see 06_kmeans_labels.csv: 363 vs 24
    terms). That is a trivial, low-information split, not a real shape
    distinction -- and it also happens to be the *coarsest* representation on
    the grid, which is why "autoselect the best setting" kept returning the
    least detailed encoding.

    Fix, in two parts:
      1. For each setting, only consider k values whose smallest cluster
         holds at least `min_cluster_fraction` of all terms (or, if using
         entropy, whose normalized entropy clears a floor). Degenerate
         "outlier vs. everything" splits are dropped from contention before
         ranking even starts.
      2. Across settings, drop the old "fewer features wins ties" rule
         (which explicitly biased toward coarser SAX words) and instead use
         balance as the tie-breaker, so a mid-resolution setting that
         produces a genuinely structured partition is preferred over a
         coarse one that only *looks* clean.
    """
    rows = []
    window_cache = make_window_cache(array_dict, settings) if use_window_cache else None

    for i, setting in enumerate(settings, start=1):
        try:
            features = build_sax_feature_matrix_fast(
                array_dict=array_dict,
                window_size=setting["window_size"],
                n_segments=setting["n_segments"],
                alphabet_size=setting["alphabet_size"],
                step=setting["step"],
                spike_features=spike_features,
                window_cache=window_cache,
            )
        except Exception as e:
            rows.append({**setting, "status": f"failed: {e}"})
            continue

        if len(features) < min_terms:
            rows.append({
                **setting,
                "status": "too_few_terms",
                "n_terms": len(features),
            })
            continue

        diag = evaluate_clustering_for_features(features, k_grid=k_grid)
        if diag.empty:
            rows.append({
                **setting,
                "status": "no_valid_k",
                "n_terms": len(features),
            })
            continue

        # --- balance-filtered selection --------------------------------
        if balance_metric == "min_frac":
            balanced = diag[diag["min_cluster_frac"] >= min_cluster_fraction]
        else:
            # a norm_entropy floor of ~0.65 already excludes "one blob + a
            # sliver" splits while still allowing moderately uneven clusters
            balanced = diag[diag["norm_entropy"] >= 0.65]

        candidate_pool = balanced if not balanced.empty else diag
        degenerate_only = balanced.empty

        best = (
            candidate_pool.sort_values(
                ["silhouette", "davies_bouldin"], ascending=[False, True]
            ).iloc[0]
        )

        rows.append({
            **setting,
            "status": "ok" if not degenerate_only else "ok_degenerate_only",
            "n_terms": len(features),
            "n_features": features.shape[1],
            "best_k": int(best["k"]),
            "silhouette": float(best["silhouette"]),
            "davies_bouldin": float(best["davies_bouldin"]),
            "calinski_harabasz": float(best["calinski_harabasz"]),
            "min_cluster_frac": float(best["min_cluster_frac"]),
            "norm_entropy": float(best["norm_entropy"]),
        })

        if i % 10 == 0:
            print(f"  evaluated {i}/{len(settings)} SAX settings")

    results = pd.DataFrame(rows)
    ok = results[results["status"] == "ok"].copy()
    degenerate = results[results["status"] == "ok_degenerate_only"].copy()
    not_ok = results[~results["status"].isin(["ok", "ok_degenerate_only"])].copy()

    # Rank by silhouette / Davies-Bouldin as before, but tie-break on balance
    # (norm_entropy, descending) instead of n_features (ascending). This is
    # the change that stops the routine from defaulting to the coarsest,
    # least-detailed SAX word on the grid.
    if not ok.empty:
        ok = ok.sort_values(
            ["silhouette", "davies_bouldin", "norm_entropy"],
            ascending=[False, True, False],
        )
    if not degenerate.empty:
        degenerate = degenerate.sort_values(
            ["silhouette", "davies_bouldin"], ascending=[False, True]
        )

    # Settings that could only produce degenerate splits at every k are kept
    # for transparency but ranked after genuinely balanced settings, never
    # silently preferred just because their silhouette number is bigger.
    return pd.concat([ok, degenerate, not_ok], ignore_index=True)


## 5. Final clustering and candidate comparison

After selecting a SAX setting, this section builds the final feature matrix and compares a small number of cluster counts. For the clinic brief, prioritize a solution that is both statistically reasonable and easy to describe.


In [84]:
def run_kmeans_final(
    features: pd.DataFrame,
    k: int,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.Series, dict]:
    """Run final KMeans on scaled SAX features."""
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    km = KMeans(n_clusters=k, random_state=random_state, n_init=FINAL_KMEANS_N_INIT)
    labels_array = km.fit_predict(X)
    labels = pd.Series(labels_array, index=features.index, name="cluster")

    metrics = {
        "k": int(k),
        "silhouette": float(silhouette_score(X, labels_array)),
        "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
        "inertia": float(km.inertia_),
    }
    return labels, metrics


def run_ward_final(features: pd.DataFrame, k: int) -> tuple[pd.Series, np.ndarray, dict]:
    """Ward hierarchical clustering for comparison."""
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    Z = linkage(X, method="ward")
    labels_array = fcluster(Z, t=k, criterion="maxclust") - 1
    labels = pd.Series(labels_array, index=features.index, name="cluster")
    metrics = {
        "k": int(k),
        "silhouette": float(silhouette_score(X, labels_array)),
        "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
    }
    return labels, Z, metrics


def cluster_member_table(labels: pd.Series) -> pd.DataFrame:
    return (
        labels.rename_axis("term")
        .reset_index()
        .sort_values(["cluster", "term"])
        .reset_index(drop=True)
    )


def cluster_size_table(labels: pd.Series) -> pd.DataFrame:
    return (
        labels.value_counts()
        .sort_index()
        .rename_axis("cluster")
        .reset_index(name="n_terms")
    )


## 5b. Hierarchical clustering add-on

This optional block runs hierarchical clustering on the same tuned sliding-SAX feature matrix used by KMeans. It is useful as a robustness check because it does not rely on random centroid initialization and produces a dendrogram that can be shown in the clinic brief.

Recommended use:

- keep KMeans as the main scalable baseline;
- use Ward hierarchical clustering as the interpretable comparison;
- compare cluster sizes and validation metrics before deciding which result to present.

In [85]:
def run_hierarchical_grid(
    features: pd.DataFrame,
    k_grid=K_GRID,
    methods=("ward", "average", "complete"),
) -> pd.DataFrame:
    """
    Evaluate hierarchical clustering across linkage methods and k values.

    Notes:
    - Ward uses Euclidean geometry and usually works best with standardized continuous features.
    - Average/complete are included as robustness checks.
    - The returned table can be compared with KMeans tuning/final metrics.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    rows = []

    for method in methods:
        try:
            Z = linkage(X, method=method)
        except Exception as e:
            rows.append({"method": method, "status": f"failed: {e}"})
            continue

        for k in k_grid:
            if not (2 <= k < len(features)):
                continue

            labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1

            # Skip degenerate cuts that do not produce at least two clusters.
            if len(np.unique(labels_array)) < 2:
                continue

            rows.append({
                "method": method,
                "k": int(k),
                "n_clusters_observed": int(len(np.unique(labels_array))),
                "silhouette": float(silhouette_score(X, labels_array)),
                "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
                "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
                "status": "ok",
            })

    results = pd.DataFrame(rows)
    ok = results[results["status"] == "ok"].copy()
    not_ok = results[results["status"] != "ok"].copy()

    if not ok.empty:
        ok = ok.sort_values(
            ["silhouette", "davies_bouldin", "calinski_harabasz"],
            ascending=[False, True, False],
        )

    return pd.concat([ok, not_ok], ignore_index=True)


def run_hierarchical_final(
    features: pd.DataFrame,
    k: int,
    method: str = "ward",
) -> tuple[pd.Series, np.ndarray, dict]:
    """Run final hierarchical clustering with a chosen linkage method and k."""
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    Z = linkage(X, method=method)
    labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1
    labels = pd.Series(labels_array, index=features.index, name="cluster")

    metrics = {
        "method": method,
        "k": int(k),
        "n_clusters_observed": int(len(np.unique(labels_array))),
        "silhouette": float(silhouette_score(X, labels_array)),
        "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
    }
    return labels, Z, metrics


## 5c. Layered (multi-resolution) clustering

A single flat `k` — whatever "autoselect" happens to land on — is exactly
what produces the "one blob + a sliver" result you were seeing. It also
throws away information: attention curves usually have real structure at
more than one resolution (e.g. "seasonal vs. non-seasonal" as a coarse
split, with "which season" or "spike-driven vs. slow-burn" as a finer split
nested inside it).

This section builds **one** Ward dendrogram on the selected SAX features and
cuts it at several heights (`LAYER_KS`, e.g. 3 / 6 / 10 clusters). Because
all cuts come from the same tree, the finer layer is guaranteed to be a
refinement of the coarser layer — every fine cluster sits inside exactly one
coarse cluster. That nesting is what makes the result "interpretable in
layers": you can present the 3-way macro grouping in the headline slide and
drill into the 10-way grouping for the appendix, without the two
contradicting each other.


In [86]:
def build_layered_clusters(
    features: pd.DataFrame,
    layer_ks: tuple[int, ...] = LAYER_KS,
    method: str = "ward",
) -> tuple[pd.DataFrame, np.ndarray]:
    """
    Cut a single hierarchical tree at multiple heights to produce nested,
    multi-resolution cluster labels (coarse -> meso -> fine).

    Returns a DataFrame indexed by term with one column per layer
    (e.g. "cluster_k3", "cluster_k6", "cluster_k10") plus the linkage matrix
    Z, so a dendrogram annotated with layer boundaries can be plotted.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    Z = linkage(X, method=method)

    layer_cols = {}
    for k in sorted(layer_ks):
        if not (2 <= k < len(features)):
            continue
        labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1
        layer_cols[f"cluster_k{k}"] = labels_array

    layered = pd.DataFrame(layer_cols, index=features.index)
    layered.index.name = "term"
    return layered, Z


def describe_layer_nesting(layered: pd.DataFrame) -> pd.DataFrame:
    """
    Sanity-check table: for each coarse-layer cluster, how many distinct
    fine-layer clusters sit inside it. If nesting is working, every
    fine-layer id maps to exactly one coarse-layer id.
    """
    cols = sorted(layered.columns, key=lambda c: int(c.split("k")[1]))
    coarsest, finest = cols[0], cols[-1]
    rows = []
    for c in sorted(layered[coarsest].unique()):
        members = layered[layered[coarsest] == c]
        rows.append({
            coarsest: c,
            "n_terms": len(members),
            "n_fine_subclusters": members[finest].nunique(),
        })
    return pd.DataFrame(rows)


## 5d. Consensus clustering (post parameter-selection)

The earlier revision of this notebook removed consensus clustering from the
core workflow. Bringing it back here, but placed *after* SAX parameter
selection rather than mixed into it, because it is solving a different
problem:

- SAX parameter tuning (section 4) asks *which representation* of the
  curves to use.
- Consensus clustering (here) asks, **for a fixed, already-chosen
  representation**, *whether a given cluster structure is stable* — i.e.
  would repeated clustering runs on resampled subsets of the terms agree on
  who belongs with whom, or is the partition an artifact of one particular
  KMeans initialization / subsample?

Method (standard evidence-accumulation / consensus clustering, Monti et al.
2003):

1. For each candidate `k`, repeatedly subsample a fraction of terms,
   cluster the subsample with KMeans, and accumulate a **co-association
   matrix** `C[i, j]` = fraction of resamples in which terms *i* and *j*
   landed in the same cluster (counting only resamples where both were
   drawn).
2. Summarize each `C` with the area under its consensus CDF — near 1.0 means
   almost every pair is either "always together" or "never together"
   (stable); near 0.5 means most pairs are ambiguous (unstable).
3. Pick `k` where that area is high and stops improving much as `k`
   increases further (the classic "delta-K" rule).
4. Build the final labels by clustering on `1 - C` (average linkage)
   instead of on the raw KMeans output, so the reported clusters are the
   ones that were actually stable across resamples, not just one lucky
   draw.
5. Report a per-term **stability score** (average co-association with its
   own cluster minus its next-best cluster) so ambiguous, "in-between"
   terms can be flagged rather than force-assigned.


In [87]:
from scipy.spatial.distance import squareform


def consensus_matrix(
    X: np.ndarray,
    k: int,
    n_resamples: int = CONSENSUS_N_RESAMPLES,
    subsample_frac: float = CONSENSUS_SUBSAMPLE_FRAC,
    random_state: int = RANDOM_STATE,
) -> np.ndarray:
    """Build the co-association (consensus) matrix for one k via subsampling + KMeans."""
    n = X.shape[0]
    rng = np.random.default_rng(random_state)
    M = np.zeros((n, n), dtype=np.float64)   # co-clustered counts
    I = np.zeros((n, n), dtype=np.float64)   # co-sampled counts
    sub_n = max(k + 1, int(round(subsample_frac * n)))

    for _ in range(n_resamples):
        idx = rng.choice(n, size=sub_n, replace=False)
        seed = int(rng.integers(0, 1_000_000))
        labels_sub = KMeans(
            n_clusters=k, n_init=10, random_state=seed
        ).fit_predict(X[idx])

        same = (labels_sub[:, None] == labels_sub[None, :]).astype(np.float64)
        M[np.ix_(idx, idx)] += same
        I[np.ix_(idx, idx)] += 1.0

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.divide(M, I, out=np.zeros_like(M), where=I > 0)
    np.fill_diagonal(C, 1.0)
    return C


def consensus_cdf_area(C: np.ndarray) -> float:
    """Area under the empirical CDF of off-diagonal consensus values (stability score)."""
    vals = C[np.triu_indices_from(C, k=1)]
    if vals.size == 0:
        return float("nan")
    vals_sorted = np.sort(vals)
    grid = np.linspace(0.0, 1.0, 101)
    cdf = np.searchsorted(vals_sorted, grid, side="right") / len(vals_sorted)
    return float(np.trapezoid(cdf, grid))


def scan_consensus_k(
    features: pd.DataFrame,
    k_range=CONSENSUS_K_RANGE,
    n_resamples: int = CONSENSUS_N_RESAMPLES,
    subsample_frac: float = CONSENSUS_SUBSAMPLE_FRAC,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, dict[int, np.ndarray]]:
    """
    Run consensus clustering across a range of k and summarize stability
    (CDF area, and its first difference) so k can be chosen for genuine
    structural stability rather than a single KMeans run's silhouette.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    matrices: dict[int, np.ndarray] = {}
    rows = []
    prev_area = None

    valid_ks = [k for k in k_range if 2 <= k < len(features)]
    for k in valid_ks:
        C = consensus_matrix(
            X, k, n_resamples=n_resamples,
            subsample_frac=subsample_frac, random_state=random_state,
        )
        matrices[k] = C
        area = consensus_cdf_area(C)
        delta = area if prev_area is None else (area - prev_area) / prev_area
        rows.append({"k": k, "consensus_cdf_area": area, "delta_area": delta})
        prev_area = area

    return pd.DataFrame(rows), matrices


def consensus_labels(C: np.ndarray, k: int, index) -> pd.Series:
    """Final cluster assignment from a consensus matrix (average linkage on 1 - C)."""
    dist = 1.0 - C
    np.fill_diagonal(dist, 0.0)
    dist = np.clip(dist, 0, None)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method="average")
    labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1
    return pd.Series(labels_array, index=index, name="cluster")


def consensus_term_stability(C: np.ndarray, labels: pd.Series) -> pd.Series:
    """
    Per-term stability score: mean co-association with its own cluster
    minus the mean co-association with the best-matching *other* cluster.
    Values near 1 = confidently, consistently clustered. Values near 0 or
    negative = ambiguous / borderline term worth flagging in the write-up.
    """
    labels_arr = labels.to_numpy()
    scores = np.zeros(len(labels_arr))
    unique_clusters = np.unique(labels_arr)

    for i in range(len(labels_arr)):
        own = labels_arr[i]
        own_mask = (labels_arr == own)
        own_mask[i] = False
        own_mean = C[i, own_mask].mean() if own_mask.any() else np.nan

        other_means = []
        for c in unique_clusters:
            if c == own:
                continue
            mask = (labels_arr == c)
            if mask.any():
                other_means.append(C[i, mask].mean())
        best_other = max(other_means) if other_means else 0.0
        scores[i] = own_mean - best_other

    return pd.Series(scores, index=labels.index, name="stability_score")


In [88]:
def plot_cluster_average_curves(
    panel_norm: pd.DataFrame,
    labels: pd.Series,
    outpath: Path,
    title: str = "Cluster-average normalized attention curves",
):
    """Plot one average curve per cluster."""
    plt.figure(figsize=(12, 6))
    for c in sorted(labels.unique()):
        members = labels[labels == c].index.intersection(panel_norm.columns)
        if len(members) == 0:
            continue
        avg = panel_norm[members].mean(axis=1, skipna=True)
        plt.plot(avg.index, avg.values, linewidth=1.8, label=f"Cluster {c} (n={len(members)})")

    plt.axhline(0, linewidth=0.8)
    plt.title(title)
    plt.ylabel("Robust z-score after denoising and detrending")
    plt.xlabel("Date")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()


def plot_dendrogram_for_features(features: pd.DataFrame, Z: np.ndarray, outpath: Path):
    plt.figure(figsize=(14, 6))
    dendrogram(Z, labels=features.index.tolist(), leaf_rotation=90, leaf_font_size=6)
    plt.title("Ward dendrogram on selected SAX features")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()


# Run pipeline

It writes all key outputs to `OUTPUT_DIR`.


In [89]:
# -----------------------------------------------------------------------------
# Add S&P 500 volatility as an extra "term" in the clustering panel
# -----------------------------------------------------------------------------

SP500_PATH = Path(r"C:\Python\CSUREMM\shock_detection\SP500_data.csv")
SP500_PRICE_COL = "price"
SP500_VOL_WINDOW = 7

def load_sp500_series(
    path: Path,
    price_col: str = SP500_PRICE_COL,
    vol_window: int = SP500_VOL_WINDOW,
) -> pd.Series:
    """
    Load S&P 500 price and construct rolling realized volatility.

    The CSV contains:
        Time, price, rolling_avg, deviation_pct, shock_up, shock_down, shock

    We use `price` because the other columns are already transformed shock
    diagnostics. The returned volatility series is then added to panel_raw
    and processed by preprocess_panel() with the Google Trends series.
    """
    df = (
        pd.read_csv(path, parse_dates=["Time"])
        .dropna(subset=["Time"])
        .drop_duplicates("Time")
        .sort_values("Time")
        .set_index("Time")
    )

    if price_col not in df.columns:
        raise ValueError(
            f"Column `{price_col}` not found. Available columns: {list(df.columns)}"
        )

    price = df[price_col].astype(float)

    log_return = np.log(price).diff()

    sp500_vol = (
        log_return
        .rolling(
            window=vol_window,
            min_periods=max(3, vol_window // 2)
        )
        .std()
    )

    sp500_vol.name = f"sp500_realized_volatility_{vol_window}d"

    return sp500_vol

In [122]:
series_raw = load_all_series(DATA_DIR)
panel_raw = build_panel(series_raw)

ZERO_SHARE_THRESHOLD = 0.50
EXPECTED_LENGTH = 1612
CORR_THRESHOLD = 0.99

# ---- 0. Initial quality report ----

quality = basic_quality_report(panel_raw)

quality.to_csv(
    OUTPUT_DIR / "01_post_stitch_quality_report.csv",
    index=False
)


# ---- 1. Drop queries with excessive zeros ----

kept_zero_queries = (
    quality
    .loc[quality["zero_share"] <= ZERO_SHARE_THRESHOLD, "query"]
    .tolist()
)

removed_zero_queries = (
    quality
    .loc[quality["zero_share"] > ZERO_SHARE_THRESHOLD, "query"]
    .tolist()
)

zero_filter_df = quality.copy()
zero_filter_df["kept_zero_share"] = (
    zero_filter_df["zero_share"] <= ZERO_SHARE_THRESHOLD
)

zero_filter_df.to_csv(
    OUTPUT_DIR / "01_zero_share_filter.csv",
    index=False
)

panel_raw = panel_raw[kept_zero_queries].copy()

print(f"Removed by zero-share filter: {len(removed_zero_queries)}")
print(f"Remaining after zero-share filter: {panel_raw.shape[1]}")


# ---- 2. Drop queries without exactly 1612 nonmissing observations ----

length_filter_df = pd.DataFrame({
    "query": panel_raw.columns,
    "n_nonmissing": [
        int(panel_raw[col].notna().sum())
        for col in panel_raw.columns
    ],
})

length_filter_df["kept_length"] = (
    length_filter_df["n_nonmissing"] == EXPECTED_LENGTH
)

length_filter_df.to_csv(
    OUTPUT_DIR / "01_length_filter_1612.csv",
    index=False
)

valid_length_queries = (
    length_filter_df
    .loc[length_filter_df["kept_length"], "query"]
    .tolist()
)

invalid_length_queries = (
    length_filter_df
    .loc[~length_filter_df["kept_length"], "query"]
    .tolist()
)

panel_raw = panel_raw[valid_length_queries].copy()

print(f"Removed by length filter: {len(invalid_length_queries)}")
print(f"Remaining after length filter: {panel_raw.shape[1]}")


# ---- 3. Drop one query from each highly correlated pair ----

corr = panel_raw.corr(
    method="pearson",
    min_periods=EXPECTED_LENGTH
).abs()

upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

to_drop = set()
duplicate_pairs = []

for col in upper.columns:
    high_corr_matches = upper.index[upper[col] > CORR_THRESHOLD].tolist()

    for match in high_corr_matches:
        if col not in to_drop and match not in to_drop:
            to_drop.add(match)

            duplicate_pairs.append({
                "kept_query": col,
                "dropped_query": match,
                "correlation": float(upper.loc[match, col]),
            })

corr_duplicate_df = pd.DataFrame(duplicate_pairs)

corr_duplicate_df.to_csv(
    OUTPUT_DIR / "01_high_correlation_dropped_pairs.csv",
    index=False
)

panel_raw = panel_raw.drop(columns=sorted(to_drop)).copy()

print(f"Dropped highly correlated queries: {len(to_drop)}")
print(f"Remaining after correlation filter: {panel_raw.shape[1]}")


# ---- 4. Save final cleaned raw panel and final quality report ----

panel_raw.to_csv(
    OUTPUT_DIR / "01_cleaned_panel_raw.csv"
)

quality_cleaned = basic_quality_report(panel_raw)

quality_cleaned.to_csv(
    OUTPUT_DIR / "01_cleaned_panel_quality_report.csv",
    index=False
)

print("Cleaned output stored as panel_raw")

# -----------------------------------------------------------------------------
# Insert S&P 500 series into panel_raw before preprocessing
# -----------------------------------------------------------------------------

sp500_series = load_sp500_series(SP500_PATH)

sp500_aligned = (
    sp500_series
    .reindex(panel_raw.index)
    .interpolate(limit_direction="both")
)

panel_raw[sp500_series.name] = sp500_aligned

print(
    f"Added {sp500_series.name} to panel_raw: "
    f"{panel_raw[sp500_series.name].notna().sum()} non-missing observations"
)

Loaded 1203 stitched series.
Removed by zero-share filter: 348
Remaining after zero-share filter: 855
Removed by length filter: 10
Remaining after length filter: 845
Dropped highly correlated queries: 16
Remaining after correlation filter: 829
Cleaned output stored as panel_raw
Added sp500_realized_volatility_7d to panel_raw: 1612 non-missing observations


In [123]:
panel_raw.drop(columns = ["sp500_realized_volatility_7d"])

,2020_election_map,2020_election_results,2022_election,2048,49ers,538,_,aaron_judge,aaron_rodgers,abcya,...,yellowstone_national_park,yellowstone_season_5,you,you_people,you_tube,young_thug,youtube,youtube.com,zillow,zoom
2022-01-01,38.0,18.000000,1.000000,11.000000,2.000000,18.000000,42.000000,0.000000,4.000000,6.000000,...,5.000000,8.000000,86.000000,62.000000,87.000000,2.000000,81.000000,87.000000,73.000000,31.000000
2022-01-02,45.0,19.000000,2.000000,12.000000,12.000000,32.000000,47.000000,0.000000,6.000000,4.000000,...,5.000000,16.000000,86.000000,67.000000,85.000000,2.000000,76.000000,80.000000,82.000000,36.000000
2022-01-03,40.0,14.000000,2.000000,20.000000,9.000000,45.000000,55.000000,1.000000,26.000000,9.000000,...,5.000000,100.000000,86.000000,67.000000,78.000000,1.000000,75.000000,76.000000,80.000000,63.000000
2022-01-04,66.0,21.000000,3.000000,29.000000,3.000000,42.000000,72.000000,0.000000,11.000000,15.000000,...,4.000000,55.000000,99.000000,74.000000,86.000000,2.000000,82.000000,74.000000,93.000000,92.000000
2022-01-05,62.0,19.000000,2.000000,28.000000,2.000000,34.000000,64.000000,0.000000,7.000000,18.000000,...,4.000000,23.000000,88.000000,76.000000,92.000000,2.000000,79.000000,78.000000,85.000000,92.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-27,0.0,17.424242,23.916495,18.535506,2.744198,16.873083,97.677372,7.278922,1.057473,5.828771,...,5.563204,1.116579,98.595529,122.882574,41.313674,1.388909,75.178230,45.722601,81.308026,49.693694
2026-05-28,0.0,12.878788,21.977320,18.535506,2.744198,16.114742,97.677372,5.823138,1.057473,6.335621,...,4.900918,1.138910,104.327828,127.378278,44.700041,1.587324,78.015145,45.722601,82.173005,55.215216
2026-05-29,0.0,17.424242,21.977320,16.909584,2.744198,17.821009,101.584467,4.658510,1.057473,6.082196,...,4.768460,1.406889,104.327828,127.378278,44.022767,1.587324,82.270516,49.109461,83.902963,46.242743
2026-05-30,0.0,15.909091,20.038144,10.080714,2.744198,14.787646,62.513518,7.861236,1.057473,1.013699,...,5.364518,1.607873,100.888449,115.389734,43.345494,1.686532,73.759773,45.722601,86.497900,32.438939


In [124]:
panel_norm, preprocessing_stages = preprocess_panel(panel_raw)
for name, stage in preprocessing_stages.items():
    stage.to_csv(OUTPUT_DIR / f"02_preprocess_{name}.csv")

## Some summary statistics analyses

In [ ]:
norm_quality = basic_quality_report(preprocessing_stages["normalized"])

norm_quality.to_csv(
    OUTPUT_DIR / "quality_report.csv",
    index=False
)

In [ ]:
term = "polls_2024"

plt.figure(figsize=(12,4))
plt.plot(preprocessing_stages["normalized"][term], label="original")
plt.plot(preprocessing_stages["clipped"][term], label="clipped")
plt.legend()

plt.savefig(OUTPUT_DIR / f"clipping_polls_2024.png", dpi=220)

In [ ]:
before = preprocessing_stages["normalized"]
after = preprocessing_stages["clipped"]

clip_fraction = ((before != after).sum() / len(before)).sort_values(ascending=False)

print(clip_fraction.head(20))

lsu_football            0.217122
football_today          0.208437
longlegs                0.186104
copa_del_rey            0.183002
march_madness           0.177419
free_covid_test_kits    0.163772
tariffs                 0.163151
nfl_standings           0.160050
ncaa_football_25        0.155707
ipl                     0.155087
syracuse_football       0.151365
olympics_2024           0.151365
mexico_baseball         0.151365
michelle_yeoh           0.147643
oscars                  0.147022
notre_dame_football     0.142680
rosh_hashanah           0.136476
duke_basketball         0.132754
nfl_games               0.132754
purdue_basketball       0.132134
dtype: float64


In [ ]:
clip_quality['std'].describe()

count    830.000000
mean       2.422971
std        0.877782
min        1.001752
25%        1.676231
50%        2.385428
75%        3.048905
max        5.111600
Name: std, dtype: float64

In [ ]:
clip_quality.sort_values("std", ascending=False).head(20)

,query,n_days_total,n_nonmissing,missing_share,zero_share,min,max,mean,std,range
267,football_today,1612,1612,0.0,0.000000,-10.0,10.0,0.890103,5.111600,20.0
420,lsu_football,1612,1612,0.0,0.181762,-10.0,10.0,0.896506,5.081842,20.0
425,march_madness,1612,1612,0.0,0.408809,-10.0,10.0,1.007167,4.812468,20.0
416,longlegs,1612,1612,0.0,0.158189,-10.0,10.0,0.954139,4.805790,20.0
152,copa_del_rey,1612,1612,0.0,0.119107,-10.0,10.0,1.244486,4.673862,20.0
528,olympics_2024,1612,1612,0.0,0.072581,-10.0,10.0,0.671817,4.650449,20.0
689,tariffs,1612,1612,0.0,0.024814,-10.0,10.0,0.260139,4.647563,20.0
483,ncaa_football_25,1612,1612,0.0,0.204094,-10.0,10.0,0.812397,4.581965,20.0
582,purdue_basketball,1612,1612,0.0,0.060794,-10.0,10.0,0.859669,4.573780,20.0
273,free_covid_test_kits,1612,1612,0.0,0.125310,-10.0,10.0,0.740918,4.520750,20.0


In [ ]:
clip_quality.sort_values("max", ascending=False).head(20)

,query,n_days_total,n_nonmissing,missing_share,zero_share,min,max,mean,std,range
826,youtube.com,1612,1612,0.0,0.024194,-10.000000,10.0,0.189794,1.703338,20.000000
0,2020_election_map,1612,1612,0.0,0.163772,-8.386474,10.0,0.453852,2.228953,18.386474
1,2020_election_results,1612,1612,0.0,0.123449,-10.000000,10.0,0.757708,2.663043,20.000000
2,2022_election,1612,1612,0.0,0.098635,-10.000000,10.0,0.776461,3.461190,20.000000
824,young_thug,1612,1612,0.0,0.014268,-10.000000,10.0,0.977940,3.070690,20.000000
4,49ers,1612,1612,0.0,0.189206,-10.000000,10.0,0.844330,3.708603,20.000000
5,538,1612,1612,0.0,0.050248,-10.000000,10.0,0.664704,3.237295,20.000000
6,_,1612,1612,0.0,0.000000,-9.385658,10.0,0.007162,2.016239,19.385658
7,aaron_judge,1612,1612,0.0,0.032258,-10.000000,10.0,0.664350,3.126528,20.000000
8,aaron_rodgers,1612,1612,0.0,0.026055,-10.000000,10.0,0.640416,2.498425,20.000000


In [ ]:
mad_before = (
    preprocessing_stages["detrended"]
    .apply(lambda s: (s - s.median()).abs().median())
)

mad_before.describe()

count    830.000000
mean       1.829184
std        1.851020
min        0.001420
25%        0.403086
50%        1.458931
75%        2.646155
max       14.760941
dtype: float64

In [ ]:
comparison = pd.DataFrame({
    "raw_std": preprocessing_stages["detrended"].std(),
    "raw_mad": preprocessing_stages["detrended"].apply(
        lambda s: (s-s.median()).abs().median()
    ),
    "norm_std": preprocessing_stages["normalized"].std(),
})

In [ ]:
comparison.describe()

,raw_std,raw_mad,norm_std
count,830.000000,830.000000,830.000000
mean,24.146771,1.829184,9.713215
std,123.541014,1.851020,20.109130
min,0.003284,0.001420,1.001752
25%,4.233714,0.403086,1.772972
50%,6.535665,1.458931,3.641340
75%,11.696664,2.646155,9.790658
max,2851.809656,14.760941,270.369417


In [ ]:
count = 0

for col in preprocessing_stages["detrended"]:
    x = preprocessing_stages["detrended"][col]

    med = x.median()
    mad = (x-med).abs().median()

    p05, p95 = x.quantile([0.05,0.95])
    spread = (p95-p05)/3.29

    zero_share = (x==0).mean()
    mad_ratio = mad/spread if spread>0 else np.nan

    if mad_ratio < 0.05:
        count += 1

print(count)

46


## back to the main pipeline

In [126]:
panel_norm = panel_norm.drop(columns=["sp500_realized_volatility_7d"])

In [127]:
panel_norm.shape

(1612, 829)

In [128]:
array_norm = panel_to_array_dict(panel_norm)

settings = make_sax_grid()
print(f"Evaluating {len(settings)} SAX settings...")

tuning_results = tune_sax_representation(array_norm, settings)
tuning_results.to_csv(OUTPUT_DIR / "03_sax_window_step_paa_alphabet_tuning.csv", index=False)

ok_results = tuning_results[tuning_results["status"] == "ok"].copy()
if ok_results.empty:
    raise ValueError("No valid SAX tuning results. Check data length and parameter grid.")

selected = ok_results.iloc[0].to_dict()
selected_params = {
    "window_size": int(selected["window_size"]),
    "step": int(selected["step"]),
    "n_segments": int(selected["n_segments"]),
    "alphabet_size": int(selected["alphabet_size"]),
    "selected_k": int(selected["best_k"]),
    "spike_features": INCLUDE_SPIKE_FEATURES,
}

pd.DataFrame([selected_params]).to_csv(OUTPUT_DIR / "04_selected_sax_parameters.csv", index=False)

print("Selected SAX parameters:")
print(json.dumps(selected_params, indent=2))
print("\nTop 10 SAX settings:")
print(ok_results.head(10).to_string(index=False))

features = build_sax_feature_matrix_fast(
    array_dict=array_norm,
    window_size=selected_params["window_size"],
    n_segments=selected_params["n_segments"],
    alphabet_size=selected_params["alphabet_size"],
    step=selected_params["step"],
    spike_features=selected_params["spike_features"],
)
features.to_csv(OUTPUT_DIR / "05_selected_sax_feature_matrix.csv")

Evaluating 52 SAX settings...
  evaluated 10/52 SAX settings
  evaluated 20/52 SAX settings
  evaluated 30/52 SAX settings
  evaluated 40/52 SAX settings
  evaluated 50/52 SAX settings
Selected SAX parameters:
{
  "window_size": 60,
  "step": 15,
  "n_segments": 16,
  "alphabet_size": 3,
  "selected_k": 2,
  "spike_features": false
}

Top 10 SAX settings:
 window_size  step  n_segments  alphabet_size status  n_terms  n_features  best_k  silhouette  davies_bouldin  calinski_harabasz  min_cluster_frac  norm_entropy
          60    15          16              3     ok      829          19       2    0.331727        1.170295         509.016401          0.352232      0.936045
          60    15          12              3     ok      829          15       2    0.313552        1.254012         472.262677          0.399276      0.970526
          60    15           8              6     ok      829          14       2    0.309778        1.344512         367.968601          0.313631      0.89732

In [129]:
# Final KMeans using the selected k from representation tuning.
labels_kmeans, metrics_kmeans = run_kmeans_final(features, k=selected_params["selected_k"])
cluster_member_table(labels_kmeans).to_csv(OUTPUT_DIR / "06_kmeans_labels.csv", index=False)
cluster_size_table(labels_kmeans).to_csv(OUTPUT_DIR / "06_kmeans_cluster_sizes.csv", index=False)
pd.DataFrame([metrics_kmeans]).to_csv(OUTPUT_DIR / "06_kmeans_metrics.csv", index=False)

# Hierarchical clustering comparison on the same selected SAX feature matrix.
hierarchical_results = run_hierarchical_grid(features, k_grid=K_GRID)
hierarchical_results.to_csv(OUTPUT_DIR / "07_hierarchical_tuning.csv", index=False)

best_hier = hierarchical_results[hierarchical_results["status"] == "ok"].iloc[0].to_dict()
labels_hier, Z_hier, metrics_hier = run_hierarchical_final(
    features,
    k=int(best_hier["k"]),
    method=str(best_hier["method"]),
)

cluster_member_table(labels_hier).to_csv(OUTPUT_DIR / "07_hierarchical_labels.csv", index=False)
cluster_size_table(labels_hier).to_csv(OUTPUT_DIR / "07_hierarchical_cluster_sizes.csv", index=False)
pd.DataFrame([metrics_hier]).to_csv(OUTPUT_DIR / "07_hierarchical_metrics.csv", index=False)

# Also save Ward with the KMeans-selected k for direct one-to-one comparison.
labels_ward, Z_ward, metrics_ward = run_ward_final(features, k=selected_params["selected_k"])
cluster_member_table(labels_ward).to_csv(OUTPUT_DIR / "07_ward_same_k_labels.csv", index=False)
cluster_size_table(labels_ward).to_csv(OUTPUT_DIR / "07_ward_same_k_cluster_sizes.csv", index=False)
pd.DataFrame([metrics_ward]).to_csv(OUTPUT_DIR / "07_ward_same_k_metrics.csv", index=False)

plot_cluster_average_curves(
    panel_norm=panel_norm,
    labels=labels_kmeans,
    outpath=OUTPUT_DIR / "08_kmeans_cluster_average_curves.png",
    title="KMeans clusters on tuned sliding-SAX features",
)
plot_cluster_average_curves(
    panel_norm=panel_norm,
    labels=labels_hier,
    outpath=OUTPUT_DIR / "08_hierarchical_cluster_average_curves.png",
    title="Hierarchical clusters on tuned sliding-SAX features",
)
plot_dendrogram_for_features(features, Z_hier, OUTPUT_DIR / "08_hierarchical_dendrogram.png")
plot_dendrogram_for_features(features, Z_ward, OUTPUT_DIR / "08_ward_same_k_dendrogram.png")

print("\nKMeans metrics:")
print(metrics_kmeans)
print("\nBest hierarchical metrics:")
print(metrics_hier)
print("\nWard same-k metrics:")
print(metrics_ward)
print("\nTop hierarchical settings:")
print(hierarchical_results.head(10).to_string(index=False))
print(f"\nOutputs written to: {OUTPUT_DIR}")


KMeans metrics:
{'k': 2, 'silhouette': 0.330634742975235, 'davies_bouldin': 1.172658948530286, 'calinski_harabasz': 509.1416976798467, 'inertia': 9749.0244140625}

Best hierarchical metrics:
{'method': 'ward', 'k': 2, 'n_clusters_observed': 2, 'silhouette': 0.32281655073165894, 'davies_bouldin': 1.194856012889694, 'calinski_harabasz': 488.019270370587}

Ward same-k metrics:
{'k': 2, 'silhouette': 0.32281655073165894, 'davies_bouldin': 1.194856012889694, 'calinski_harabasz': 488.019270370587}

Top hierarchical settings:
  method  k  n_clusters_observed  silhouette  davies_bouldin  calinski_harabasz status
    ward  2                    2    0.322817        1.194856         488.019270     ok
 average  2                    2    0.316512        1.212566         481.519357     ok
 average  3                    3    0.291429        1.301290         264.684568     ok
 average  4                    4    0.218230        1.258052         181.979966     ok
 average  5                    5    0.1

## Evaluate candidate numbers of clusters for the selected SAX representation

In [130]:
K_CANDIDATES = [2, 3, 4, 5, 6]

k_evaluation = evaluate_clustering_for_features(
    features,
    k_grid=[2, 3, 4, 5, 6],
    silhouette_sample_size=None,
)

k_evaluation.to_csv(
    OUTPUT_DIR / "AA_selected_representation_k_evaluation.csv",
    index=False,
)

print("\nCluster evaluation for selected SAX representation:")
print(
    k_evaluation[
        [
            "k",
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz",
            "min_cluster_frac",
            "norm_entropy",
        ]
    ].to_string(index=False)
)


Cluster evaluation for selected SAX representation:
 k  silhouette  davies_bouldin  calinski_harabasz  min_cluster_frac  norm_entropy
 2    0.331063        1.170295         509.016401          0.352232      0.936045
 3    0.182431        1.824230         340.865392          0.290712      0.991340
 4    0.169688        1.872298         255.136139          0.179735      0.984190
 5    0.151215        1.874516         229.026195          0.129071      0.979629
 6    0.135450        1.917685         199.841757          0.113390      0.985827


## Consensus clustering on the selected representation

Runs on top of the `features` matrix built in the cell above, using the
SAX setting chosen by the fixed (balance-aware) selection rule.

- **Layered clustering** cuts one Ward tree at `LAYER_KS` to give nested
  coarse/meso/fine cluster columns (`09_layered_clusters.csv`).
- **Consensus clustering** scans `CONSENSUS_K_RANGE`, reports stability
  (`10_consensus_k_scan.csv`), and produces final consensus-based labels
  plus a per-term stability score for the chosen k
  (`11_consensus_labels.csv`).


In [131]:
# --- 6。 Consensus clustering scan across k -----------------------------------
print(f"\nRunning consensus clustering ({CONSENSUS_N_RESAMPLES} resamples per k, "
      f"k in {list(CONSENSUS_K_RANGE)})...")
consensus_scan, consensus_matrices = scan_consensus_k(
    features,
    k_range=CONSENSUS_K_RANGE,
    n_resamples=CONSENSUS_N_RESAMPLES,
    subsample_frac=CONSENSUS_SUBSAMPLE_FRAC,
)
consensus_scan.to_csv(OUTPUT_DIR / "10_consensus_k_scan.csv", index=False)
print(consensus_scan.to_string(index=False))

# Pick k with the highest consensus CDF area whose gain over the previous k
# has already leveled off (classic delta-K rule), rather than always taking
# the single highest silhouette k as before.
stable_candidates = consensus_scan[consensus_scan["delta_area"].abs() < 0.02]
if not stable_candidates.empty:
    consensus_k = int(stable_candidates.iloc[0]["k"])
else:
    consensus_k = int(consensus_scan.loc[consensus_scan["consensus_cdf_area"].idxmax(), "k"])
print(f"\nSelected consensus k = {consensus_k}")

C_selected = consensus_matrices[consensus_k]
labels_consensus = consensus_labels(C_selected, consensus_k, features.index)
stability = consensus_term_stability(C_selected, labels_consensus)

consensus_table = (
    cluster_member_table(labels_consensus)
    .merge(stability.rename("stability_score").reset_index().rename(columns={"index": "term"}),
           on="term", how="left")
    .sort_values(["cluster", "stability_score"], ascending=[True, False])
    .reset_index(drop=True)
)
consensus_table.to_csv(OUTPUT_DIR / "11_consensus_labels.csv", index=False)
cluster_size_table(labels_consensus).to_csv(OUTPUT_DIR / "11_consensus_cluster_sizes.csv", index=False)

print(f"\nConsensus cluster sizes (k={consensus_k}):")
print(cluster_size_table(labels_consensus).to_string(index=False))
print("\nLowest-stability (most ambiguous) terms:")
print(consensus_table.sort_values("stability_score").head(10).to_string(index=False))

plot_cluster_average_curves(
    panel_norm=panel_norm,
    labels=labels_consensus,
    outpath=OUTPUT_DIR / "11_consensus_cluster_average_curves.png",
    title=f"Consensus clusters (k={consensus_k}) on tuned sliding-SAX features",
)

print(f"\nConsensus outputs written to: {OUTPUT_DIR}")



Running consensus clustering (200 resamples per k, k in [2, 3, 4, 5, 6, 7, 8, 9, 10])...
 k  consensus_cdf_area  delta_area
 2            0.461273    0.461273
 3            0.664171    0.439864
 4            0.743285    0.119117
 5            0.793409    0.067437
 6            0.823450    0.037863
 7            0.850646    0.033026
 8            0.871025    0.023957
 9            0.883364    0.014167
10            0.894652    0.012779

Selected consensus k = 9

Consensus cluster sizes (k=9):
 cluster  n_terms
       0      106
       1      105
       2       98
       3       99
       4      105
       5       46
       6       68
       7       84
       8      118

Lowest-stability (most ambiguous) terms:
                term  cluster  stability_score
                suns        2        -0.400773
         adam_levine        2        -0.299250
          miami_heat        2        -0.295658
               fedex        0        -0.290109
     wakanda_forever        2        -0.28689

In [132]:
# ------------------------------------------------------------------
# Plot the normalized time series of the top 20 most stable terms
# in each consensus cluster
# ------------------------------------------------------------------

import matplotlib.pyplot as plt

TOP_N = 20

# Use the normalized panel from the preprocessing pipeline
panel_plot = preprocessing_stages["normalized"]

for cluster_id in sorted(consensus_table["cluster"].unique()):

    top_terms = (
        consensus_table.loc[
            consensus_table["cluster"] == cluster_id,
            ["term", "stability_score"]
        ]
        .head(TOP_N)["term"]
        .tolist()
    )

    plt.figure(figsize=(14, 5))

    for term in top_terms:
        if term in panel_plot.columns:
            plt.plot(
                panel_plot.index,
                panel_plot[term],
                linewidth=1.2,
                alpha=0.8,
                label=term,
            )

    plt.title(f"Cluster {cluster_id}: Top {TOP_N} Most Stable Words (Normalized)")
    plt.xlabel("Time")
    plt.ylabel("Normalized Search Interest")
    plt.grid(alpha=0.3)
    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"cluster_{cluster_id}_top{TOP_N}_normalized.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

# Internal Validation

In [136]:
# ------------------------------------------------------------------
# Internal validation of consensus clusters
# ------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------------
# Inputs expected from previous cells:
#   consensus_table: columns = term, cluster, stability_score
#   labels_consensus: pd.Series or dict-like, index/keys = term
#   features: selected SAX feature matrix, index = term
#   panel_norm: normalized time-series panel, columns = term
#   OUTPUT_DIR
# ------------------------------------------------------------------

VALIDATION_DIR = OUTPUT_DIR / "consensus_cluster_internal_validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Internal validation of consensus clusters
# Focus: SAX-space cohesion, centroid similarity, stability,
# and descriptive cluster trajectories.
# ------------------------------------------------------------------

def average_pairwise_cosine_similarity(X: np.ndarray) -> float:
    if X.shape[0] < 2:
        return np.nan

    sim = cosine_similarity(X)
    upper = sim[np.triu_indices_from(sim, k=1)]

    return float(np.mean(upper))


def centroid_similarity_summary(X: np.ndarray) -> dict:
    if X.shape[0] < 2:
        return {
            "mean_centroid_dist": np.nan,
            "median_centroid_dist": np.nan,
            "max_centroid_dist": np.nan,
            "mean_centroid_cosine": np.nan,
            "median_centroid_cosine": np.nan,
            "min_centroid_cosine": np.nan,
        }

    centroid = X.mean(axis=0, keepdims=True)

    dist = np.linalg.norm(X - centroid, axis=1)

    cos = cosine_similarity(X, centroid).ravel()

    return {
        "mean_centroid_dist": float(np.mean(dist)),
        "median_centroid_dist": float(np.median(dist)),
        "max_centroid_dist": float(np.max(dist)),
        "mean_centroid_cosine": float(np.mean(cos)),
        "median_centroid_cosine": float(np.median(cos)),
        "min_centroid_cosine": float(np.min(cos)),
    }

In [137]:
# ------------------------------------------------------------------
# Align terms across consensus labels, SAX features, and time-series panel
# ------------------------------------------------------------------

common_terms = (
    consensus_table["term"]
    .loc[consensus_table["term"].isin(features.index)]
    .loc[lambda s: s.isin(panel_norm.columns)]
    .unique()
)

labels_df = (
    consensus_table
    .loc[consensus_table["term"].isin(common_terms)]
    [["term", "cluster", "stability_score"]]
    .copy()
)

features_valid = features.loc[common_terms].copy()

X_scaled = pd.DataFrame(
    StandardScaler().fit_transform(features_valid),
    index=features_valid.index,
    columns=features_valid.columns,
)

panel_valid = panel_norm.loc[:, common_terms].copy()

In [138]:
# ------------------------------------------------------------------
# Cluster-level validation summary
# ------------------------------------------------------------------

validation_rows = []
term_centroid_rows = []

for cluster_id in sorted(labels_df["cluster"].unique()):

    terms = (
        labels_df
        .loc[labels_df["cluster"] == cluster_id, "term"]
        .tolist()
    )

    X_cluster_df = X_scaled.loc[terms]
    X_cluster = X_cluster_df.to_numpy()

    stability_cluster = (
        labels_df
        .loc[labels_df["cluster"] == cluster_id, "stability_score"]
        .dropna()
    )

    q25 = stability_cluster.quantile(0.25)
    q75 = stability_cluster.quantile(0.75)

    centroid_summary = centroid_similarity_summary(X_cluster)

    row = {
        "cluster": cluster_id,
        "n_terms": len(terms),

        # SAX-space cohesion
        "avg_sax_cosine_similarity": average_pairwise_cosine_similarity(X_cluster),

        # Compactness / centroid similarity in clustered feature space
        **centroid_summary,

        # Consensus stability distribution
        "stability_min": float(stability_cluster.min()),
        "stability_q25": float(q25),
        "stability_median": float(stability_cluster.median()),
        "stability_q75": float(q75),
        "stability_iqr": float(q75 - q25),
        "stability_mean": float(stability_cluster.mean()),
    }

    validation_rows.append(row)

    # Term-level distance/similarity to cluster centroid
    centroid = X_cluster_df.mean(axis=0).to_numpy().reshape(1, -1)

    term_dist = np.linalg.norm(
        X_cluster_df.to_numpy() - centroid,
        axis=1,
    )

    term_cos = cosine_similarity(
        X_cluster_df.to_numpy(),
        centroid,
    ).ravel()

    temp = pd.DataFrame({
        "term": terms,
        "cluster": cluster_id,
        "distance_to_cluster_centroid": term_dist,
        "cosine_to_cluster_centroid": term_cos,
    })

    term_centroid_rows.append(temp)


cluster_validation = pd.DataFrame(validation_rows)

cluster_validation.to_csv(
    VALIDATION_DIR / "cluster_internal_validation_summary.csv",
    index=False,
)

print("\nCluster internal validation summary:")
print(cluster_validation.to_string(index=False))


term_centroid_validation = (
    pd.concat(term_centroid_rows, ignore_index=True)
    .merge(
        labels_df[["term", "stability_score"]],
        on="term",
        how="left",
    )
    .sort_values(["cluster", "cosine_to_cluster_centroid"], ascending=[True, False])
)

term_centroid_validation.to_csv(
    VALIDATION_DIR / "term_centroid_validation.csv",
    index=False,
)


Cluster internal validation summary:
 cluster  n_terms  avg_sax_cosine_similarity  mean_centroid_dist  median_centroid_dist  max_centroid_dist  mean_centroid_cosine  median_centroid_cosine  min_centroid_cosine  stability_min  stability_q25  stability_median  stability_q75  stability_iqr  stability_mean
       0      106                   0.328138            2.763829              2.709289           4.842326              0.577583                0.577228             0.126640      -0.290109       0.424381          0.682993       0.749188       0.324806        0.566659
       1      105                   0.306606            2.500428              2.365707           4.605344              0.559311                0.588866            -0.104375      -0.024029       0.424855          0.570094       0.599523       0.174668        0.486379
       2       98                   0.489265            2.703730              2.592820           4.972559              0.702887                0.707234          

In [140]:
# ------------------------------------------------------------------
# Cluster interpretation figures:
#   1. SAX segment centroid profiles
#   2. Average symbol-frequency profiles
#   3. Top 10 prototype queries by centroid cosine similarity
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# Prepare aligned feature + label data
# ------------------------------------------------------------------

labels_df = (
    consensus_table[["term", "cluster", "stability_score"]]
    .loc[consensus_table["term"].isin(features.index)]
    .copy()
)

features_labeled = features.loc[labels_df["term"]].copy()
features_labeled["term"] = features_labeled.index
features_labeled = features_labeled.merge(
    labels_df,
    on="term",
    how="left",
)


# ------------------------------------------------------------------
# Identify SAX feature groups
# ------------------------------------------------------------------

seg_cols = [
    c for c in features.columns
    if c.startswith("sax_seg_mean_")
]

symbol_cols = [
    c for c in features.columns
    if c.startswith("symbol_share_")
]


# ------------------------------------------------------------------
# Option 1: plot cluster centroid profiles across SAX/PAA segments
# ------------------------------------------------------------------

segment_profiles = (
    features_labeled
    .groupby("cluster")[seg_cols]
    .mean()
)

segment_profiles.to_csv(
    VALIDATION_DIR / "cluster_sax_segment_centroid_profiles.csv"
)

for cluster_id, row in segment_profiles.iterrows():

    x = np.arange(1, len(seg_cols) + 1)

    plt.figure(figsize=(8, 4))

    plt.plot(
        x,
        row.values,
        marker="o",
        linewidth=2,
    )

    plt.xticks(x)
    plt.xlabel("PAA segment position within SAX window")
    plt.ylabel("Average symbolic level")
    plt.title(f"Cluster {cluster_id}: SAX Segment Centroid Profile")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        VALIDATION_DIR / f"cluster_{cluster_id}_sax_segment_centroid.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()




In [142]:
# ------------------------------------------------------------------
# Option 2: plot average symbol-frequency profiles
# ------------------------------------------------------------------

symbol_profiles = (
    features_labeled
    .groupby("cluster")[symbol_cols]
    .mean()
)

symbol_profiles.to_csv(
    VALIDATION_DIR / "cluster_symbol_share_profiles.csv"
)

for cluster_id, row in symbol_profiles.iterrows():

    symbol_ids = [
        int(c.replace("symbol_share_", ""))
        for c in symbol_cols
    ]

    plt.figure(figsize=(6, 4))

    plt.bar(
        symbol_ids,
        row.values,
    )

    plt.xticks(symbol_ids)
    plt.xlabel("SAX symbol")
    plt.ylabel("Average share")
    plt.title(f"Cluster {cluster_id}: Average SAX Symbol Distribution")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        VALIDATION_DIR / f"cluster_{cluster_id}_symbol_share_profile.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


In [143]:
# ------------------------------------------------------------------
# Option 3: top 10 prototype queries by centroid cosine similarity
# ------------------------------------------------------------------

# Recompute centroid cosine if term_centroid_validation is not already available.
if "term_centroid_validation" not in globals():

    X_scaled = pd.DataFrame(
        StandardScaler().fit_transform(features),
        index=features.index,
        columns=features.columns,
    )

    proto_rows = []

    for cluster_id in sorted(labels_df["cluster"].unique()):

        terms = (
            labels_df
            .loc[labels_df["cluster"] == cluster_id, "term"]
            .tolist()
        )

        X_cluster = X_scaled.loc[terms]
        centroid = X_cluster.mean(axis=0).to_numpy().reshape(1, -1)

        centroid_cos = cosine_similarity(
            X_cluster.to_numpy(),
            centroid,
        ).ravel()

        temp = pd.DataFrame({
            "term": terms,
            "cluster": cluster_id,
            "cosine_to_cluster_centroid": centroid_cos,
        })

        proto_rows.append(temp)

    term_centroid_validation = (
        pd.concat(proto_rows, ignore_index=True)
        .merge(
            labels_df[["term", "stability_score"]],
            on="term",
            how="left",
        )
    )


top10_prototypes = (
    term_centroid_validation
    .sort_values(
        ["cluster", "cosine_to_cluster_centroid"],
        ascending=[True, False],
    )
    .groupby("cluster")
    .head(10)
    .reset_index(drop=True)
)

top10_prototypes.to_csv(
    VALIDATION_DIR / "top10_prototype_queries_by_cluster.csv",
    index=False,
)

print("\nTop 10 prototype queries by cluster:")
print(
    top10_prototypes[
        [
            "cluster",
            "term",
            "cosine_to_cluster_centroid",
            "stability_score",
        ]
    ].to_string(index=False)
)


Top 10 prototype queries by cluster:
 cluster                         term  cosine_to_cluster_centroid  stability_score
       0                   tsla_stock                    0.922391         0.757243
       0                        sonic                    0.880531         0.763379
       0                         tsla                    0.870834         0.759113
       0                   ny_yankees                    0.866832         0.759357
       0               andrew_wiggins                    0.863510         0.760097
       0               world_cup_2022                    0.854128         0.686517
       0        golden_state_warriors                    0.837335         0.710215
       0            tesla_stock_price                    0.830437         0.763719
       0                dennis_rodman                    0.825444         0.743252
       0                  tesla_stock                    0.812858         0.760717
       1                     nfl_news            

# S&P 500

In [144]:
# -----------------------------------------------------------------------------
# External S&P 500 benchmark + cluster index analysis
# Do NOT add S&P 500 to the clustering panel.
# Use it after clustering as an external outcome/benchmark.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


# -----------------------------------------------------------------------------
# Load S&P 500 price, returns, and realized volatility
# -----------------------------------------------------------------------------

SP500_PATH = Path(r"C:\Python\CSUREMM\shock_detection\SP500_data.csv")
SP500_PRICE_COL = "price"
SP500_VOL_WINDOW = 7

MARKET_DIR = OUTPUT_DIR / "consensus_cluster_sp500_analysis"
MARKET_DIR.mkdir(parents=True, exist_ok=True)


def load_sp500_market_panel(
    path: Path,
    price_col: str = SP500_PRICE_COL,
    vol_window: int = SP500_VOL_WINDOW,
) -> pd.DataFrame:

    df = (
        pd.read_csv(path, parse_dates=["Time"])
        .dropna(subset=["Time"])
        .drop_duplicates("Time")
        .sort_values("Time")
        .set_index("Time")
    )

    if price_col not in df.columns:
        raise ValueError(
            f"Column `{price_col}` not found. Available columns: {list(df.columns)}"
        )

    price = df[price_col].astype(float)

    log_return = np.log(price).diff()

    realized_vol = (
        log_return
        .rolling(
            window=vol_window,
            min_periods=max(3, vol_window // 2),
        )
        .std()
    )

    market = pd.DataFrame({
        "sp500_price": price,
        "sp500_log_return": log_return,
        f"sp500_realized_volatility_{vol_window}d": realized_vol,
    })

    return market


market = load_sp500_market_panel(SP500_PATH)


# -----------------------------------------------------------------------------
# Construct one Google Trends index per consensus cluster
# Uses normalized Google Trends series only.
# S&P 500 should NOT be included as a cluster member.
# -----------------------------------------------------------------------------

def build_cluster_indices(
    panel_norm: pd.DataFrame,
    consensus_table: pd.DataFrame,
    weighting: str = "stability",
    top_n: int | None = None,
) -> pd.DataFrame:

    index_dict = {}

    for cluster_id in sorted(consensus_table["cluster"].unique()):

        cluster_terms_df = (
            consensus_table
            .loc[consensus_table["cluster"] == cluster_id]
            .copy()
        )

        cluster_terms_df = cluster_terms_df[
            cluster_terms_df["term"].isin(panel_norm.columns)
        ]

        if top_n is not None:
            cluster_terms_df = (
                cluster_terms_df
                .sort_values("stability_score", ascending=False)
                .head(top_n)
            )

        terms = cluster_terms_df["term"].tolist()

        if len(terms) == 0:
            continue

        X = panel_norm[terms].copy()

        if weighting == "equal":
            index_series = X.mean(axis=1)

        elif weighting == "stability":
            w = cluster_terms_df["stability_score"].astype(float).copy()

            # Shift weights to be nonnegative within cluster
            w = w - w.min()

            if w.sum() == 0:
                w = pd.Series(np.ones(len(w)), index=w.index)

            w = w / w.sum()

            index_series = X.mul(w.to_numpy(), axis=1).sum(axis=1)

        else:
            raise ValueError("weighting must be either 'equal' or 'stability'")

        index_series.name = f"cluster_{cluster_id}_index"
        index_dict[index_series.name] = index_series

    return pd.DataFrame(index_dict)


cluster_indices_equal = build_cluster_indices(
    panel_norm=panel_norm,
    consensus_table=consensus_table,
    weighting="equal",
    top_n=None,
)

cluster_indices_stability = build_cluster_indices(
    panel_norm=panel_norm,
    consensus_table=consensus_table,
    weighting="stability",
    top_n=None,
)

cluster_indices_top20 = build_cluster_indices(
    panel_norm=panel_norm,
    consensus_table=consensus_table,
    weighting="stability",
    top_n=20,
)

cluster_indices_equal.to_csv(MARKET_DIR / "cluster_indices_equal_weight.csv")
cluster_indices_stability.to_csv(MARKET_DIR / "cluster_indices_stability_weighted.csv")
cluster_indices_top20.to_csv(MARKET_DIR / "cluster_indices_top20_stability_weighted.csv")

In [145]:
# -----------------------------------------------------------------------------
# Align cluster indices with S&P 500 market data
# -----------------------------------------------------------------------------

target_col = f"sp500_realized_volatility_{SP500_VOL_WINDOW}d"

analysis_df = (
    cluster_indices_stability
    .join(market, how="inner")
    .dropna()
)

analysis_df.to_csv(MARKET_DIR / "cluster_indices_with_sp500.csv")


# -----------------------------------------------------------------------------
# Shape comparison: cluster indices vs S&P realized volatility
# Normalize each series for visual comparison only.
# -----------------------------------------------------------------------------

def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()


plot_df = analysis_df.copy()

for col in cluster_indices_stability.columns:
    plot_df[col] = zscore(plot_df[col])

plot_df[target_col] = zscore(plot_df[target_col])


for col in cluster_indices_stability.columns:

    plt.figure(figsize=(14, 5))

    plt.plot(
        plot_df.index,
        plot_df[col],
        label=col,
        linewidth=1.5,
    )

    plt.plot(
        plot_df.index,
        plot_df[target_col],
        label=target_col,
        linewidth=1.5,
        alpha=0.8,
    )

    plt.axhline(0, linewidth=1, alpha=0.5)

    plt.title(f"Shape Comparison: {col} vs S&P 500 Realized Volatility")
    plt.xlabel("Time")
    plt.ylabel("Z-scored value")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        MARKET_DIR / f"shape_comparison_{col}_vs_sp500_vol.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


# -----------------------------------------------------------------------------
# Lead-lag correlation test
# Corr(cluster_index_t, market_target_{t+h})
# h > 0 means the cluster index leads the S&P variable.
# -----------------------------------------------------------------------------

def lead_lag_correlation(
    df: pd.DataFrame,
    index_cols: list[str],
    target_col: str,
    max_lag: int = 30,
) -> pd.DataFrame:

    rows = []

    for index_col in index_cols:

        for h in range(-max_lag, max_lag + 1):

            corr = df[index_col].corr(
                df[target_col].shift(-h)
            )

            rows.append({
                "cluster_index": index_col,
                "horizon_days": h,
                "correlation": corr,
            })

    return pd.DataFrame(rows)


lead_lag_corr = lead_lag_correlation(
    analysis_df,
    index_cols=list(cluster_indices_stability.columns),
    target_col=target_col,
    max_lag=30,
)

lead_lag_corr.to_csv(
    MARKET_DIR / "lead_lag_correlations_cluster_indices_vs_sp500_vol.csv",
    index=False,
)


# Plot lead-lag correlation curves
for index_col in cluster_indices_stability.columns:

    temp = lead_lag_corr[
        lead_lag_corr["cluster_index"] == index_col
    ]

    plt.figure(figsize=(8, 4))

    plt.plot(
        temp["horizon_days"],
        temp["correlation"],
        marker="o",
        linewidth=1.5,
    )

    plt.axhline(0, linewidth=1, alpha=0.5)
    plt.axvline(0, linewidth=1, alpha=0.5)

    plt.title(f"Lead-Lag Correlation: {index_col} vs S&P Volatility")
    plt.xlabel("Horizon h: corr(index_t, market_{t+h})")
    plt.ylabel("Correlation")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        MARKET_DIR / f"lead_lag_corr_{index_col}_vs_sp500_vol.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


# -----------------------------------------------------------------------------
# Predictive regressions:
# target_{t+h} = alpha + beta * cluster_index_t
# -----------------------------------------------------------------------------

def predictive_regression_scan(
    df: pd.DataFrame,
    index_cols: list[str],
    target_col: str,
    horizons: list[int],
) -> pd.DataFrame:

    rows = []

    for index_col in index_cols:

        for h in horizons:

            y = df[target_col].shift(-h)
            X = df[[index_col]]

            reg_df = (
                pd.concat([X, y.rename("target")], axis=1)
                .dropna()
            )

            if len(reg_df) < 30:
                continue

            X_mat = reg_df[[index_col]].to_numpy()
            y_vec = reg_df["target"].to_numpy()

            model = LinearRegression()
            model.fit(X_mat, y_vec)

            y_hat = model.predict(X_mat)

            rows.append({
                "cluster_index": index_col,
                "target": target_col,
                "horizon_days": h,
                "n_obs": len(reg_df),
                "beta": float(model.coef_[0]),
                "intercept": float(model.intercept_),
                "r2": float(r2_score(y_vec, y_hat)),
                "rmse": float(mean_squared_error(y_vec, y_hat) ** 0.5),
                "corr": float(pd.Series(reg_df[index_col]).corr(reg_df["target"])),
            })

    return pd.DataFrame(rows)


prediction_results = predictive_regression_scan(
    analysis_df,
    index_cols=list(cluster_indices_stability.columns),
    target_col=target_col,
    horizons=[1, 7, 14, 30, 60],
)

prediction_results.to_csv(
    MARKET_DIR / "predictive_regressions_cluster_indices_vs_sp500_vol.csv",
    index=False,
)

print("\nTop predictive cluster-index results:")
print(
    prediction_results
    .sort_values(["r2", "corr"], ascending=[False, False])
    .head(20)
    .to_string(index=False)
)


Top predictive cluster-index results:
  cluster_index                       target  horizon_days  n_obs      beta  intercept       r2     rmse      corr
cluster_2_index sp500_realized_volatility_7d             1   1101  0.001198   0.006406 0.306514 0.003706  0.553637
cluster_2_index sp500_realized_volatility_7d             7   1095  0.001182   0.006443 0.298171 0.003735  0.546050
cluster_5_index sp500_realized_volatility_7d             7   1095  0.000466   0.007556 0.117104 0.004189  0.342205
cluster_5_index sp500_realized_volatility_7d             1   1101  0.000423   0.007634 0.096520 0.004231  0.310677
cluster_0_index sp500_realized_volatility_7d             1   1101  0.000555   0.007674 0.096348 0.004231  0.310399
cluster_0_index sp500_realized_volatility_7d             7   1095  0.000545   0.007702 0.092780 0.004246  0.304599
cluster_2_index sp500_realized_volatility_7d            14   1088  0.000497   0.007646 0.052602 0.004349  0.229351
cluster_1_index sp500_realized_volatility

In [146]:
# -----------------------------------------------------------------------------
# Optional: compare index construction choices
# equal-weight vs stability-weight vs top-20 stability-weight
# -----------------------------------------------------------------------------

index_sets = {
    "equal": cluster_indices_equal,
    "stability": cluster_indices_stability,
    "top20_stability": cluster_indices_top20,
}

all_prediction_results = []

for index_name, index_panel in index_sets.items():

    temp_df = (
        index_panel
        .join(market, how="inner")
        .dropna()
    )

    temp_results = predictive_regression_scan(
        temp_df,
        index_cols=list(index_panel.columns),
        target_col=target_col,
        horizons=[1, 7, 14, 30, 60],
    )

    temp_results["index_construction"] = index_name

    all_prediction_results.append(temp_results)


all_prediction_results = pd.concat(
    all_prediction_results,
    ignore_index=True,
)

all_prediction_results.to_csv(
    MARKET_DIR / "predictive_regressions_all_index_constructions.csv",
    index=False,
)

print("\nBest results across index-construction methods:")
print(
    all_prediction_results
    .sort_values(["r2", "corr"], ascending=[False, False])
    .head(30)
    .to_string(index=False)
)

print(f"\nMarket comparison and prediction outputs written to: {MARKET_DIR}")


Best results across index-construction methods:
  cluster_index                       target  horizon_days  n_obs     beta  intercept       r2     rmse     corr index_construction
cluster_2_index sp500_realized_volatility_7d             1   1101 0.001198   0.006406 0.306514 0.003706 0.553637          stability
cluster_2_index sp500_realized_volatility_7d             7   1095 0.001182   0.006443 0.298171 0.003735 0.546050          stability
cluster_2_index sp500_realized_volatility_7d             1   1101 0.001214   0.006331 0.271690 0.003798 0.521239              equal
cluster_2_index sp500_realized_volatility_7d             7   1095 0.001191   0.006378 0.261646 0.003831 0.511513              equal
cluster_2_index sp500_realized_volatility_7d             1   1101 0.000578   0.007656 0.205451 0.003967 0.453267    top20_stability
cluster_2_index sp500_realized_volatility_7d             7   1095 0.000565   0.007689 0.196143 0.003997 0.442880    top20_stability
cluster_5_index sp500_reali

In [147]:
# Inspect Cluster 2 terms
print(
    consensus_table
    .loc[consensus_table["cluster"] == 2]
    .sort_values("stability_score", ascending=False)
    .head(40)
    .to_string(index=False)
)

                       term  cluster  stability_score
          midterm_elections        2         0.462935
        the_weather_channel        2         0.459911
            victoria_secret        2         0.455755
             brendan_fraser        2         0.454094
             denver_broncos        2         0.422912
severe_thunderstorm_warning        2         0.422239
             texas_shooting        2         0.413547
                disney_plus        2         0.407260
            weather_channel        2         0.393131
             trump_shooting        2         0.384841
               amazon_stock        2         0.378264
      election_results_2022        2         0.377829
                   big_lots        2         0.376388
                    sephora        2         0.375354
             cindy_williams        2         0.374359
            minecraft_movie        2         0.371090
                johnny_depp        2         0.337768
            mexico_baseball 

In [149]:
# ------------------------------------------------------------
# Out-of-sample test with train-only scaling + expanding window
# ------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

target_col = "sp500_realized_volatility_7d"
index_col = "cluster_2_index"
h = 1

df = analysis_df[[index_col, target_col]].copy()
df["target_future"] = df[target_col].shift(-h)
df = df.dropna()

initial_train_frac = 0.70
initial_train_size = int(len(df) * initial_train_frac)

pred_rows = []

for t in range(initial_train_size, len(df)):

    train = df.iloc[:t]
    test = df.iloc[[t]]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[[index_col]])
    X_test = scaler.transform(test[[index_col]])

    y_train = train["target_future"]
    y_test = test["target_future"].iloc[0]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)[0]

    pred_rows.append({
        "date": test.index[0],
        "actual": y_test,
        "predicted": y_pred,
        "beta": model.coef_[0],
        "intercept": model.intercept_,
    })

oos_pred = pd.DataFrame(pred_rows).set_index("date")

test_r2 = r2_score(oos_pred["actual"], oos_pred["predicted"])
test_rmse = mean_squared_error(oos_pred["actual"], oos_pred["predicted"]) ** 0.5
test_corr = oos_pred["predicted"].corr(oos_pred["actual"])

print("Expanding-window out-of-sample test")
print("Cluster:", index_col)
print("Horizon:", h)
print("Initial train n:", initial_train_size)
print("Test n:", len(oos_pred))
print("OOS R2:", test_r2)
print("OOS RMSE:", test_rmse)
print("OOS corr:", test_corr)

Expanding-window out-of-sample test
Cluster: cluster_2_index
Horizon: 1
Initial train n: 770
Test n: 331
OOS R2: 0.42018212132256894
OOS RMSE: 0.004183697295516675
OOS corr: 0.6634088736616761


In [150]:
# ------------------------------------------------------------------
# External validation: OOS prediction tests for all cluster indices
# ------------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------------
# Expanding-window prediction helper
# ------------------------------------------------------------------

def expanding_window_oos(
    df: pd.DataFrame,
    predictors: list[str],
    target_col: str,
    horizon: int = 1,
    initial_train_frac: float = 0.70,
) -> tuple[pd.DataFrame, dict]:

    work = df[predictors + [target_col]].copy()
    work["target_future"] = work[target_col].shift(-horizon)
    work = work.dropna()

    initial_train_size = int(len(work) * initial_train_frac)

    pred_rows = []

    for t in range(initial_train_size, len(work)):

        train = work.iloc[:t]
        test = work.iloc[[t]]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(train[predictors])
        X_test = scaler.transform(test[predictors])

        y_train = train["target_future"]
        y_test = test["target_future"].iloc[0]

        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)[0]

        pred_rows.append({
            "date": test.index[0],
            "actual": y_test,
            "predicted": y_pred,
        })

    pred = pd.DataFrame(pred_rows).set_index("date")

    metrics = {
        "horizon_days": horizon,
        "predictors": "+".join(predictors),
        "initial_train_n": initial_train_size,
        "test_n": len(pred),
        "oos_r2": r2_score(pred["actual"], pred["predicted"]),
        "oos_rmse": mean_squared_error(pred["actual"], pred["predicted"]) ** 0.5,
        "oos_corr": pred["actual"].corr(pred["predicted"]),
    }

    return pred, metrics


# ------------------------------------------------------------------
# Test 1: run same OOS test for every cluster index
# ------------------------------------------------------------------

target_col = "sp500_realized_volatility_7d"
cluster_index_cols = [
    c for c in analysis_df.columns
    if c.startswith("cluster_") and c.endswith("_index")
]

horizons = [1, 7, 14, 30]

all_oos_metrics = []
all_oos_predictions = {}

for h in horizons:

    for index_col in cluster_index_cols:

        pred, metrics = expanding_window_oos(
            df=analysis_df,
            predictors=[index_col],
            target_col=target_col,
            horizon=h,
            initial_train_frac=0.70,
        )

        metrics["cluster_index"] = index_col
        metrics["model"] = "cluster_only"

        all_oos_metrics.append(metrics)
        all_oos_predictions[(index_col, h, "cluster_only")] = pred

oos_metrics = pd.DataFrame(all_oos_metrics)

oos_metrics.to_csv(
    MARKET_DIR / "oos_all_clusters_cluster_only.csv",
    index=False,
)

print("\nOOS results: cluster-only models")
print(
    oos_metrics
    .sort_values(["horizon_days", "oos_r2"], ascending=[True, False])
    .to_string(index=False)
)



OOS results: cluster-only models
 horizon_days      predictors  initial_train_n  test_n    oos_r2  oos_rmse  oos_corr   cluster_index        model
            1 cluster_2_index              770     331  0.420182  0.004184  0.663409 cluster_2_index cluster_only
            1 cluster_0_index              770     331  0.052567  0.005348  0.253397 cluster_0_index cluster_only
            1 cluster_1_index              770     331  0.030071  0.005411  0.193450 cluster_1_index cluster_only
            1 cluster_5_index              770     331  0.005558  0.005479  0.168350 cluster_5_index cluster_only
            1 cluster_8_index              770     331  0.001485  0.005490  0.054779 cluster_8_index cluster_only
            1 cluster_3_index              770     331  0.001312  0.005491  0.061707 cluster_3_index cluster_only
            1 cluster_7_index              770     331 -0.002825  0.005502  0.058755 cluster_7_index cluster_only
            1 cluster_4_index              770     331

## Tests for Cluster 2 only

In [154]:
# ------------------------------------------------------------------
# Clinic brief tests: Cluster 2 external validation
# ------------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CLINIC_DIR = MARKET_DIR / "clinic_brief_cluster2"
CLINIC_DIR.mkdir(parents=True, exist_ok=True)

target_col = "sp500_realized_volatility_7d"
cluster_id = 2
horizons = [1, 7]
cluster_col = f"cluster_{cluster_id}_index"


# ------------------------------------------------------------------
# Helper: fixed split OOS regression
# ------------------------------------------------------------------

def fixed_split_oos(
    df,
    predictors,
    target_col,
    horizon=1,
    train_frac=0.70,
):
    cols = list(dict.fromkeys(predictors + [target_col]))

    work = df.loc[:, cols].copy()

    # if duplicate column names exist, keep the first occurrence
    work = work.loc[:, ~work.columns.duplicated()]

    target_series = work[target_col].squeeze()
    work["target_future"] = target_series.shift(-horizon)

    work = work.dropna()

    split = int(len(work) * train_frac)

    train = work.iloc[:split]
    test = work.iloc[split:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[predictors])
    X_test = scaler.transform(test[predictors])

    y_train = train["target_future"]
    y_test = test["target_future"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    pred = pd.Series(
        model.predict(X_test),
        index=test.index,
        name="predicted",
    )

    metrics = {
        "predictors": "+".join(predictors),
        "horizon_days": horizon,
        "train_n": len(train),
        "test_n": len(test),
        "oos_r2": r2_score(y_test, pred),
        "oos_rmse": mean_squared_error(y_test, pred) ** 0.5,
        "oos_corr": pred.corr(y_test),
    }

    pred_df = pd.DataFrame({
        "actual": y_test,
        "predicted": pred,
    })

    return pred_df, metrics

In [155]:
# ------------------------------------------------------------------
# 1. Incremental benchmark test:
#    Vol_{t+h} ~ Vol_t
#    vs.
#    Vol_{t+h} ~ Vol_t + Cluster 2 index
# ------------------------------------------------------------------

incremental_rows = []
prediction_store = {}

for h in horizons:

    pred_bench, m_bench = fixed_split_oos(
        df=analysis_df,
        predictors=[target_col],
        target_col=target_col,
        horizon=h,
    )

    m_bench["model"] = "vol_persistence"

    pred_cluster, m_cluster = fixed_split_oos(
        df=analysis_df,
        predictors=[cluster_col],
        target_col=target_col,
        horizon=h,
    )

    m_cluster["model"] = "cluster2_only"

    pred_incremental, m_incremental = fixed_split_oos(
        df=analysis_df,
        predictors=[target_col, cluster_col],
        target_col=target_col,
        horizon=h,
    )

    m_incremental["model"] = "vol_persistence_plus_cluster2"

    prediction_store[(h, "benchmark")] = pred_bench
    prediction_store[(h, "cluster2_only")] = pred_cluster
    prediction_store[(h, "incremental")] = pred_incremental

    incremental_rows.extend([m_bench, m_cluster, m_incremental])

clinic_oos = pd.DataFrame(incremental_rows)

benchmark_r2 = (
    clinic_oos
    .loc[clinic_oos["model"] == "vol_persistence", ["horizon_days", "oos_r2", "oos_rmse"]]
    .rename(columns={
        "oos_r2": "benchmark_oos_r2",
        "oos_rmse": "benchmark_oos_rmse",
    })
)

clinic_oos = clinic_oos.merge(
    benchmark_r2,
    on="horizon_days",
    how="left",
)

clinic_oos["incremental_oos_r2"] = (
    clinic_oos["oos_r2"] - clinic_oos["benchmark_oos_r2"]
)

clinic_oos["rmse_improvement"] = (
    clinic_oos["benchmark_oos_rmse"] - clinic_oos["oos_rmse"]
)

clinic_oos.to_csv(
    CLINIC_DIR / "cluster2_incremental_oos_tests.csv",
    index=False,
)

print("\nCluster 2 incremental OOS tests:")
print(clinic_oos.to_string(index=False))



Cluster 2 incremental OOS tests:
                                  predictors  horizon_days  train_n  test_n   oos_r2  oos_rmse  oos_corr                         model  benchmark_oos_r2  benchmark_oos_rmse  incremental_oos_r2  rmse_improvement
                sp500_realized_volatility_7d             1      770     331 0.862220  0.002039  0.928737               vol_persistence          0.862220            0.002039            0.000000          0.000000
                             cluster_2_index             1      770     331 0.424137  0.004169  0.659778                 cluster2_only          0.862220            0.002039           -0.438083         -0.002130
sp500_realized_volatility_7d+cluster_2_index             1      770     331 0.879621  0.001906  0.937905 vol_persistence_plus_cluster2          0.862220            0.002039            0.017400          0.000133
                sp500_realized_volatility_7d             7      766     329 0.133818  0.005122  0.382202               vol

In [156]:
# ------------------------------------------------------------------
# 2. Actual vs predicted plots for Cluster 2 at h = 1 and h = 7
# ------------------------------------------------------------------

for h in horizons:

    pred = prediction_store[(h, "cluster2_only")]

    plt.figure(figsize=(14, 5))

    plt.plot(
        pred.index,
        pred["actual"],
        label="Actual future S&P volatility",
        linewidth=1.5,
    )

    plt.plot(
        pred.index,
        pred["predicted"],
        label="Predicted by Cluster 2 index",
        linewidth=1.5,
    )

    plt.title(f"Cluster 2 OOS Prediction of S&P 500 Volatility, h={h}")
    plt.xlabel("Time")
    plt.ylabel(target_col)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        CLINIC_DIR / f"cluster2_actual_vs_predicted_h{h}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

In [157]:
# ------------------------------------------------------------------
# 3. Cluster 2 prototype terms
#    A. by centroid cosine
#    B. by stability score
# ------------------------------------------------------------------

cluster2_by_centroid = (
    term_centroid_validation
    .loc[term_centroid_validation["cluster"] == cluster_id]
    .sort_values("cosine_to_cluster_centroid", ascending=False)
    .head(30)
)

cluster2_by_stability = (
    consensus_table
    .loc[consensus_table["cluster"] == cluster_id]
    .sort_values("stability_score", ascending=False)
    .head(30)
)

cluster2_by_centroid.to_csv(
    CLINIC_DIR / "cluster2_top30_by_centroid_cosine.csv",
    index=False,
)

cluster2_by_stability.to_csv(
    CLINIC_DIR / "cluster2_top30_by_stability.csv",
    index=False,
)

print("\nCluster 2 prototype terms by centroid cosine:")
print(cluster2_by_centroid.to_string(index=False))

print("\nCluster 2 prototype terms by stability:")
print(cluster2_by_stability.to_string(index=False))


# ------------------------------------------------------------------
# 4. Robustness table:
#    Equal-weight vs stability-weight vs top-20 stability-weight
#    Cluster 2 only
# ------------------------------------------------------------------

index_panels = {
    "equal": cluster_indices_equal,
    "stability": cluster_indices_stability,
    "top20_stability": cluster_indices_top20,
}

robustness_rows = []

for construction_name, index_panel in index_panels.items():

    cluster_col = f"cluster_{cluster_id}_index"

    if cluster_col not in index_panel.columns:
        continue

    temp_df = (
        index_panel[[cluster_col]]
        .join(market, how="inner")
        .dropna()
    )

    for h in horizons:

        pred, metrics = fixed_split_oos(
            df=temp_df,
            predictors=[cluster_col],
            target_col=target_col,
            horizon=h,
        )

        metrics["cluster"] = cluster_id
        metrics["index_construction"] = construction_name

        robustness_rows.append(metrics)

cluster2_robustness = pd.DataFrame(robustness_rows)

cluster2_robustness.to_csv(
    CLINIC_DIR / "cluster2_index_construction_robustness.csv",
    index=False,
)

print("\nCluster 2 index-construction robustness:")
print(
    cluster2_robustness
    .sort_values(["horizon_days", "oos_r2"], ascending=[True, False])
    .to_string(index=False)
)

print(f"\nClinic brief outputs written to: {CLINIC_DIR}")


Cluster 2 prototype terms by centroid cosine:
                    term  cluster  distance_to_cluster_centroid  cosine_to_cluster_centroid  stability_score
                fortnite        2                      2.150556                    0.964308         0.237954
              lia_thomas        2                      3.523294                    0.945201         0.281574
            jenna_ortega        2                      3.570462                    0.933077         0.116899
         freddie_freeman        2                      2.224996                    0.919334         0.228942
           tornado_watch        2                      3.914765                    0.919130         0.287359
                 shakira        2                      2.875743                    0.901555         0.263738
               lady_gaga        2                      2.417059                    0.900953         0.278517
                 broncos        2                      2.331892                  

## Tests for all clusters

In [ ]:
# ------------------------------------------------------------------
# Test 2: benchmark model using volatility persistence only
# Vol_{t+h} ~ Vol_t
# ------------------------------------------------------------------

benchmark_metrics = []
benchmark_predictions = {}

for h in horizons:

    pred, metrics = expanding_window_oos(
        df=analysis_df,
        predictors=[target_col],
        target_col=target_col,
        horizon=h,
        initial_train_frac=0.70,
    )

    metrics["cluster_index"] = "benchmark_vol_persistence"
    metrics["model"] = "vol_persistence"

    benchmark_metrics.append(metrics)
    benchmark_predictions[h] = pred

benchmark_metrics = pd.DataFrame(benchmark_metrics)

benchmark_metrics.to_csv(
    MARKET_DIR / "oos_benchmark_vol_persistence.csv",
    index=False,
)

print("\nOOS benchmark results: volatility persistence")
print(benchmark_metrics.to_string(index=False))

In [ ]:
# ------------------------------------------------------------------
# Test 3: incremental model
# Vol_{t+h} ~ Vol_t + ClusterIndex_t
# ------------------------------------------------------------------

incremental_metrics = []
incremental_predictions = {}

for h in horizons:

    for index_col in cluster_index_cols:

        pred, metrics = expanding_window_oos(
            df=analysis_df,
            predictors=[target_col, index_col],
            target_col=target_col,
            horizon=h,
            initial_train_frac=0.70,
        )

        metrics["cluster_index"] = index_col
        metrics["model"] = "vol_persistence_plus_cluster"

        incremental_metrics.append(metrics)
        incremental_predictions[(index_col, h)] = pred

incremental_metrics = pd.DataFrame(incremental_metrics)

incremental_metrics.to_csv(
    MARKET_DIR / "oos_vol_persistence_plus_clusters.csv",
    index=False,
)

print("\nOOS results: volatility persistence + cluster index")
print(
    incremental_metrics
    .sort_values(["horizon_days", "oos_r2"], ascending=[True, False])
    .to_string(index=False)
)

In [ ]:
# ------------------------------------------------------------------
# Test 4: incremental OOS R2 over benchmark
# ------------------------------------------------------------------

comparison = incremental_metrics.merge(
    benchmark_metrics[
        ["horizon_days", "oos_r2", "oos_rmse", "oos_corr"]
    ].rename(columns={
        "oos_r2": "benchmark_oos_r2",
        "oos_rmse": "benchmark_oos_rmse",
        "oos_corr": "benchmark_oos_corr",
    }),
    on="horizon_days",
    how="left",
)

comparison["incremental_oos_r2"] = (
    comparison["oos_r2"] - comparison["benchmark_oos_r2"]
)

comparison["rmse_improvement"] = (
    comparison["benchmark_oos_rmse"] - comparison["oos_rmse"]
)

comparison.to_csv(
    MARKET_DIR / "oos_incremental_cluster_value_over_benchmark.csv",
    index=False,
)

print("\nIncremental predictive value over volatility persistence:")
print(
    comparison
    .sort_values(["horizon_days", "incremental_oos_r2"], ascending=[True, False])
    [
        [
            "cluster_index",
            "horizon_days",
            "oos_r2",
            "benchmark_oos_r2",
            "incremental_oos_r2",
            "oos_corr",
            "benchmark_oos_corr",
            "rmse_improvement",
        ]
    ]
    .to_string(index=False)
)


In [ ]:
# ------------------------------------------------------------------
# Test 5: plot actual vs predicted for best cluster-only model
# ------------------------------------------------------------------

best_row = (
    oos_metrics
    .sort_values("oos_r2", ascending=False)
    .iloc[0]
)

best_cluster = best_row["cluster_index"]
best_h = int(best_row["horizon_days"])

best_pred = all_oos_predictions[(best_cluster, best_h, "cluster_only")]

plt.figure(figsize=(14, 5))

plt.plot(
    best_pred.index,
    best_pred["actual"],
    label="Actual future S&P volatility",
    linewidth=1.5,
)

plt.plot(
    best_pred.index,
    best_pred["predicted"],
    label=f"Predicted by {best_cluster}",
    linewidth=1.5,
)

plt.title(
    f"Out-of-Sample Prediction: {best_cluster}, horizon={best_h} day"
)
plt.xlabel("Time")
plt.ylabel(target_col)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    MARKET_DIR / f"oos_actual_vs_predicted_{best_cluster}_h{best_h}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ------------------------------------------------------------------
# Test 6: plot incremental model for best cluster over benchmark
# ------------------------------------------------------------------

best_inc_row = (
    comparison
    .sort_values("incremental_oos_r2", ascending=False)
    .iloc[0]
)

best_inc_cluster = best_inc_row["cluster_index"]
best_inc_h = int(best_inc_row["horizon_days"])

best_inc_pred = incremental_predictions[(best_inc_cluster, best_inc_h)]
benchmark_pred = benchmark_predictions[best_inc_h]

plt.figure(figsize=(14, 5))

plt.plot(
    best_inc_pred.index,
    best_inc_pred["actual"],
    label="Actual future S&P volatility",
    linewidth=1.5,
)

plt.plot(
    benchmark_pred.index,
    benchmark_pred["predicted"],
    label="Benchmark: volatility persistence",
    linewidth=1.3,
    alpha=0.8,
)

plt.plot(
    best_inc_pred.index,
    best_inc_pred["predicted"],
    label=f"Benchmark + {best_inc_cluster}",
    linewidth=1.5,
)

plt.title(
    f"Incremental OOS Prediction: {best_inc_cluster}, horizon={best_inc_h} day"
)
plt.xlabel("Time")
plt.ylabel(target_col)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    MARKET_DIR / f"oos_incremental_actual_vs_predicted_{best_inc_cluster}_h{best_inc_h}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ------------------------------------------------------------------
# Test 7: inspect prototype terms for best predictive cluster
# ------------------------------------------------------------------

best_cluster_id = int(
    best_cluster
    .replace("cluster_", "")
    .replace("_index", "")
)

print(f"\nTop terms in best predictive cluster {best_cluster_id}:")
print(
    consensus_table
    .loc[consensus_table["cluster"] == best_cluster_id]
    .sort_values("stability_score", ascending=False)
    .head(40)
    .to_string(index=False)
)

print(f"\nOOS testing outputs written to: {MARKET_DIR}")

## Interpretation notes for reporting

Use the following outputs for the clinic brief:

- `03_sax_window_step_paa_alphabet_tuning.csv`
  SAX settings, now with `min_cluster_frac` / `norm_entropy` columns and a
  `status` of `ok` (balanced split found), `ok_degenerate_only` (every k for
  this setting only produced an outlier-vs-rest split), or a failure reason.

- `04_selected_sax_parameters.csv`
  The chosen `window_size`, `step`, PAA word length `w`, and alphabet size
  `a` — now selected by balance-filtered silhouette rather than raw
  silhouette, so it should no longer default to the single coarsest
  representation on the grid.

- `05_selected_sax_feature_matrix.csv`
  Query-level feature matrix used for all downstream clustering (KMeans,
  hierarchical, layered, consensus).

- `06_kmeans_labels.csv` / `06_kmeans_cluster_sizes.csv`, `07_hierarchical_*`,
  `07_ward_same_k_*`
  Flat KMeans / Ward comparisons at the selected k, kept for continuity with
  the previous version of this notebook.

- `09_layered_clusters.csv`, `09_layered_nesting_check.csv`,
  `09_layered_dendrogram.png`
  **Nested, multi-resolution clusters** (coarse/meso/fine cuts of one Ward
  tree at `LAYER_KS`). Use the coarse layer for the headline grouping and
  the fine layer to drill into subtypes within each coarse group — this is
  the "interpretable layers" view.

- `10_consensus_k_scan.csv`
  Stability (consensus CDF area) for each candidate k. Use this, not a
  single silhouette number, to justify the choice of k in the brief.

- `11_consensus_labels.csv`, `11_consensus_cluster_sizes.csv`,
  `11_consensus_cluster_average_curves.png`
  **Consensus clustering** results: cluster assignments built from repeated
  KMeans on resampled subsets rather than one KMeans run, plus a
  per-term `stability_score` (low/negative = ambiguous term, worth flagging
  or excluding from a "clean" narrative rather than force-assigned).

### What changed in this revision, and why

Diagnosis of the original run:

- The selection rule picked the setting with the single highest silhouette
  across the whole grid. That setting was `window_size=365, n_segments=8,
  alphabet_size=3` at `k=2` — the **coarsest** representation on the grid.
- At `k=2` it split 387 terms into 363 vs. 24 (KMeans) and 383 vs. 4
  (Ward): one dominant cluster plus a small set of outliers. That kind of
  split is easy to separate geometrically (hence the high silhouette /
  Davies-Bouldin), but it is not a meaningful shape distinction — it is
  "normal terms vs. extreme-magnitude terms," and it happens at nearly every
  coarse setting, which is why coarse settings kept winning.
- The original tie-break rule for equally-good settings explicitly preferred
  **fewer features**, i.e. more compression — actively pushing the
  autoselection toward less detail, which is the opposite of what "better
  separate into interpretable layers" needs.
- Consensus clustering had been removed from the workflow entirely, so there
  was no mechanism to check whether a chosen k was actually a stable
  structure or an artifact of one KMeans run.

Fixes applied:

1. **Balance-aware selection** (`cluster_balance_metrics`, updated
   `tune_sax_representation`): every k is now scored with `min_cluster_frac`
   and `norm_entropy` in addition to silhouette/DB/CH. Degenerate
   outlier-vs-rest splits are excluded from contention before ranking, and
   the tie-break rule no longer rewards coarser (fewer-feature)
   representations.
2. **Layered clustering** (`build_layered_clusters`): one Ward tree, cut at
   several heights, so coarse and fine groupings are nested and mutually
   consistent instead of relying on one flat, autoselected k.
3. **Consensus clustering after parameter selection** (`consensus_matrix`,
   `scan_consensus_k`, `consensus_labels`, `consensus_term_stability`):
   restored, but scoped correctly — it runs once, on the SAX representation
   already chosen in section 4, to validate/select k by structural
   stability and to flag ambiguous terms, rather than being folded into the
   representation search itself.

### Efficiency revision summary (unchanged from the previous version)

The computationally expensive part is the grid search over sliding-SAX
representations. Cached NumPy arrays and sliding windows, vectorized PAA,
and sampled silhouette scoring during tuning are all retained. Consensus
clustering is the new expensive step (`CONSENSUS_N_RESAMPLES` KMeans fits
per k); it is deliberately run once on the final selected feature matrix,
not inside the SAX tuning grid search, to keep the overall runtime
reasonable.
